# Data cleaning & preparation :

In [ ]:
import pandas as pd
import numpy as np


In [104]:
# paths (defined once)
RAW = "../Data/Raw/"
PROCESSED = "../Data/Processed/"

orders = pd.read_csv(RAW + "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW + "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW + "olist_customers_dataset.csv")
payments = pd.read_csv(RAW + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW + "olist_order_reviews_dataset.csv")
sellers = pd.read_csv(RAW + "olist_sellers_dataset.csv")
products = pd.read_csv(RAW + "olist_products_dataset.csv")
product_name_translation = pd.read_csv(RAW + "product_category_name_translation.csv")
geolocation = pd.read_csv(RAW + "olist_geolocation_dataset.csv")

In [105]:
#dataframes copies for cleaning

orders_clean = orders.copy()
order_items_clean = order_items.copy()
customers_clean = customers.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
sellers_clean = sellers.copy()
products_clean = products.copy()
geolocation_clean = geolocation.copy()

## orders Table :

### Missing values :

In [106]:
#Missing Values overview

df = orders_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

missing_overview = missing_overview[missing_overview["missing_count"] > 0].sort_values(by="missing_percent",ascending=False)
display(missing_overview)

,missing_count,missing_percent
order_delivered_customer_date,2965,2.981668
order_delivered_carrier_date,1783,1.793023
order_approved_at,160,0.160899


In [107]:
#investigating missing values

#check order status of missing values
df = orders_clean

missing_customer_delivery = df[df["order_delivered_customer_date"].isna()]

print("--- order status of missing customer delivery ---")
print(missing_customer_delivery["order_status"].value_counts(dropna=False))

missing_carrier_delivery = df[df["order_delivered_carrier_date"].isna()]

print("\n--- order status of missing carrier delivery ---")
print(missing_carrier_delivery["order_status"].value_counts(dropna=False))

missing_approved = df[df["order_approved_at"].isna()]

print("\n--- order status of missing approved ---")
print(missing_approved["order_status"].value_counts(dropna=False))


--- order status of missing customer delivery ---
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

--- order status of missing carrier delivery ---
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

--- order status of missing approved ---
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64


In [108]:
#checking inconsistencies 

#for customer delivery
display(missing_customer_delivery[missing_customer_delivery["order_status"] == "delivered"].head())

#for carrier delivery
display(missing_carrier_delivery[missing_carrier_delivery["order_status"] == "delivered"].head())

#for approved
display(missing_approved[missing_approved["order_status"] == "delivered"].head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaN,2017-11-20 19:44:47,2017-11-14 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:47,2017-03-01 13:25:33,2017-03-17 00:00:00
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:31,NaN,2017-02-23 09:01:52,2017-03-02 10:05:06,2017-03-21 00:00:00
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:47,NaN,2017-02-22 16:25:25,2017-03-01 08:07:38,2017-03-17 00:00:00
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:35,NaN,2017-02-22 11:23:10,2017-03-09 07:28:47,2017-03-31 00:00:00
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:55,NaN,2017-02-22 11:23:11,2017-03-02 11:09:19,2017-03-20 00:00:00


#### cleaning decision :

No rows will be deleted, and no missing timestamps will be imputed at this stage, because Most missing delivery timestamps belong to orders whose statuses indicate that they may not have reached the relevent delivery stage.

The identified inconsistencies will be documented for further investigation. inconsistencies are those that have wrong status for example in this table some orders have delivered status but the customer delivered timestamp did not recorded.
but we couldn't drop all those inconsistencies because we would lose some important data, so that's why we documented them

### duplicates :

In [109]:
df = orders_clean

#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

print(
    "Duplicate order_id:",
    df["order_id"].duplicated().sum()
)

Exact duplicates: 0
Duplicate order_id: 0


#### Duplicate Records

**Investigation:** Exact duplicate rows and duplicate `order_id` values were checked.

**Decision:** No duplicate records were identified based on these checks.

**Action:** No rows were removed.

**Validation:** The duplicate checks confirmed that the identified conditions were not present in the current dataset.

### Data types :

In [110]:
df = orders_clean

display(df.dtypes)

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

In [111]:
# Find the orders that are null after converting to datetime but they weren't null before that

parsed = pd.to_datetime(
    df["order_delivered_customer_date"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

problem_mask = df["order_delivered_customer_date"].notna() & parsed.isna()

print("Number of problematic values:", problem_mask.sum())

display(df.loc[problem_mask, ["order_id", "order_delivered_customer_date"]])



Number of problematic values: 2


,order_id,order_delivered_customer_date
33,d887b52c6516beb39e8cd44a5f8b60f7,2018-02-22 007:55
169,b2f3763408103bc7173bb36ed02dbeea,2018-06-18:0 23:07:33


In [112]:
# Fix porblematic values in order_delivered_customer_date

df["order_delivered_customer_date"] = (
    df["order_delivered_customer_date"].replace({
        "2018-02-22 007:55": "2018-02-22 00:07:55",
        "2018-06-18:0 23:07:33": "2018-06-18 23:07:33"
    })
)

# Convert date columns to datetime

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="%Y-%m-%d %H:%M:%S"
    )

# Validate data types
display(df.dtypes)

# Check missing values after conversion
display(df[date_cols].isna().sum())

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

#### Data Types conclusion :

**Investigation:** The data types of all columns were inspected. The timestamp columns were initially stored as strings.

**Cleaning Decision:** The timestamp columns were converted to `datetime`. Two incorrectly formatted values in `order_delivered_customer_date` were corrected before conversion.

**Validation:** Data types and missing-value counts were checked after conversion to ensure the changes were applied correctly.

### invalid & suspicious values :

In [113]:
df = orders_clean

#checking invalid values in order_status
display(df["order_status"].value_counts(dropna=False))

#after runnig the cell there is no invalid values

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [114]:
#check chronological inconsistencies

checks = {
    "purchase_after_approval": (
        "order_purchase_timestamp",
        "order_approved_at"
    ),
    "purchase_after_carrier_delivery": (
        "order_purchase_timestamp",
        "order_delivered_carrier_date"
    ),
    "purchase_after_customer_delivery": (
        "order_purchase_timestamp",
        "order_delivered_customer_date"
    ),
    "approval_after_carrier_delivery": (
        "order_approved_at",
        "order_delivered_carrier_date"
    ),
    "approval_after_customer_delivery": (
        "order_approved_at",
        "order_delivered_customer_date"
    ),
    "customer_delivery_before_carrier_delivery": (
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    )
}

results = []

for name, (date1, date2) in checks.items():

    mask = (
        df[date1].notna()
        & df[date2].notna()
        & (df[date1] > df[date2])
    )

    results.append({
        "check": name,
        "records": mask.sum(),
        "percentage": round(mask.mean() * 100, 3)
    })

anomaly_summary = pd.DataFrame(results)

display(anomaly_summary)

,check,records,percentage
0,purchase_after_approval,0,0.000
1,purchase_after_carrier_delivery,166,0.167
2,purchase_after_customer_delivery,0,0.000
3,approval_after_carrier_delivery,1359,1.367
4,approval_after_customer_delivery,61,0.061
5,customer_delivery_before_carrier_delivery,23,0.023


In [115]:
#handling chronological anomalies
#every anomaly gets a flag column, so the processed file records which rows were affected

# 1. Carrier date before purchase date
# most gaps are under an hour (clock / recording artefact), so values are kept and flagged
df["flag_carrier_before_purchase"] = (
    df["order_delivered_carrier_date"].notna()
    & (df["order_delivered_carrier_date"] < df["order_purchase_timestamp"])
)

# 2. Customer delivery date before carrier date
df["flag_customer_before_carrier"] = (
    df["order_delivered_customer_date"].notna()
    & df["order_delivered_carrier_date"].notna()
    & (df["order_delivered_customer_date"] < df["order_delivered_carrier_date"])
)

# 3. Approval date after customer delivery date
df["flag_approval_after_customer"] = (
    df["order_approved_at"].notna()
    & df["order_delivered_customer_date"].notna()
    & (df["order_approved_at"] > df["order_delivered_customer_date"])
)

#turn impossible values to NaT (2 and 3 only)
df.loc[df["flag_customer_before_carrier"], "order_delivered_carrier_date"] = pd.NaT
df.loc[df["flag_approval_after_customer"], "order_approved_at"] = pd.NaT

# 4. Approval date after carrier date: kept and flagged (possible recording delay)
df["flag_approval_after_carrier"] = (
    df["order_approved_at"].notna()
    & df["order_delivered_carrier_date"].notna()
    & (df["order_approved_at"] > df["order_delivered_carrier_date"])
)

# gap size of the kept carrier-before-purchase records (hours)
gap_hours = (
    df.loc[df["flag_carrier_before_purchase"], "order_purchase_timestamp"]
    - df.loc[df["flag_carrier_before_purchase"], "order_delivered_carrier_date"]
).dt.total_seconds() / 3600
print("carrier before purchase, gap in hours:")
display(gap_hours.describe().round(2))

carrier before purchase, gap in hours:


count     166.00
mean       26.04
std       318.92
min         0.01
25%         0.23
50%         0.60
75%         1.02
max      4109.10
dtype: float64

In [116]:
#validation
checks = {
    "carrier_before_purchase (kept, flagged)": df["order_delivered_carrier_date"] < df["order_purchase_timestamp"],
    "customer_before_carrier": df["order_delivered_customer_date"] < df["order_delivered_carrier_date"],
    "approval_after_delivery": df["order_approved_at"] > df["order_delivered_customer_date"],
    "approval_after_carrier_delivery (kept, flagged)": df["order_approved_at"] > df["order_delivered_carrier_date"]
}

for name, condition in checks.items():
    print(f"{name}: {condition.fillna(False).sum()} records")

flag_cols = [c for c in df.columns if c.startswith("flag_")]
print("\nflag columns:")
display(df[flag_cols].sum())

carrier_before_purchase (kept, flagged): 166 records
customer_before_carrier: 0 records
approval_after_delivery: 0 records
approval_after_carrier_delivery (kept, flagged): 1298 records

flag columns:


flag_carrier_before_purchase     166
flag_customer_before_carrier      23
flag_approval_after_customer      61
flag_approval_after_carrier     1298
dtype: int64

#### Timestamp Anomalies

The following decisions were made to handle chronological inconsistencies in the `orders` dataset. Every case gets a boolean `flag_*` column, so the processed file records exactly which rows were affected.

| Issue | Count | Severity | Decision | Flag column |
|---|---:|---|---|---|
| Carrier before purchase | 166 | Likely recording artefact | Keep both dates and flag. | `flag_carrier_before_purchase` |
| Customer delivery before carrier | 23 | Impossible | Set `order_delivered_carrier_date` to `NaT`, keeping the customer delivery date unchanged. | `flag_customer_before_carrier` |
| Approval after customer delivery | 61 | Impossible | Set `order_approved_at` to `NaT`. | `flag_approval_after_customer` |
| Approval after carrier | 1,298 (after the fixes above) | Mild | Keep both dates and flag. Exclude them only from approval-to-carrier durations. | `flag_approval_after_carrier` |

**Reasoning**

- **Carrier before purchase:** the median gap is only 0.6 hours and 75% of the gaps are under about 1 hour, which points to clock or recording differences rather than a wrong event. The values are therefore kept and flagged, the same policy as "approval after carrier". Analyses of the purchase-to-carrier step should exclude flagged rows.
- **Customer delivery before carrier:** Since the customer delivery date is retained as the more reliable event, the carrier date is set to `NaT`. This decision affects only 23 records.
- **Approval after customer delivery:** This sequence contradicts the expected order process. The approval timestamp is considered unreliable and set to `NaT`, while the customer delivery date is retained.
- **Approval after carrier:** This inconsistency may result from a recording delay rather than an incorrect event timestamp. Because the cause cannot be established confidently, both dates are retained and the affected records are flagged. They are excluded only from calculations of the duration between approval and carrier.

**Important:** No orders are deleted as part of this cleaning step. Only 84 timestamp values (23 carrier dates and 61 approval dates) are changed to `NaT`. The remaining data are preserved.

In [117]:
#check date range

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

display(df[date_cols].agg(["min", "max"]))

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
min,2016-09-04 21:15:19,2016-09-15 12:16:38,2016-10-08 10:34:01,2016-10-11 13:46:32,2016-09-30
max,2018-10-17 17:30:18,2018-09-03 17:40:06,2018-09-11 19:48:28,2018-10-17 13:22:46,2018-11-12


#### Invalid & Suspicious Values conclusion

**Investigation:** The `order_status` values were checked for unexpected categories, and timestamp columns were examined for chronological inconsistencies between order purchase, approval, carrier handover, and customer delivery events. The date ranges were also inspected to understand the temporal coverage of the dataset.

**Findings:** Several timestamp inconsistencies were identified, where recorded events did not follow the expected chronological order.

**Decision:** Impossible sequences were marked as missing. Ambiguous cases, which are most likely recording artefacts, were retained. Every case was flagged in its own column.

**Action:** 84 timestamp values were set to `NaT` and 4 flag columns were added. No order records were deleted.

**Validation:** The chronological checks were used to identify the affected records and verify the results of the cleaning operations. Flagged records should be excluded only from the durations they affect.

### preparation

In [118]:
# 1. Extract order date components
df["order_year"] = df["order_purchase_timestamp"].dt.year
df["order_month"] = df["order_purchase_timestamp"].dt.month
df["order_day"] = df["order_purchase_timestamp"].dt.day
df["order_weekday"] = df["order_purchase_timestamp"].dt.day_name()



#finding Delivery time
df["delivery_duration_days"] = (
    df["order_delivered_customer_date"]
    - df["order_purchase_timestamp"]
).dt.total_seconds() / 86400


#Number of days delayed in delivery (calendar days)
#the estimated date has no time part (00:00), so the delivery timestamp is reduced to its date first,
#otherwise an order delivered on the estimated day would count as late
df["delivery_delay_days"] = (
    df["order_delivered_customer_date"].dt.normalize()
    - df["order_estimated_delivery_date"]
).dt.days.astype("Int64")

#late flag (missing when there is no delivery date)
df["is_late"] = (df["delivery_delay_days"] > 0).astype("boolean")


#durations of intermediate steps
df["approval_hours"] = (
    df["order_approved_at"]
    - df["order_purchase_timestamp"]
).dt.total_seconds() / 3600

df["carrier_to_customer_days"] = (
    df["order_delivered_customer_date"]
    - df["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

#comparison with the old timestamp-based definition
old_late = df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]
delivered = df["order_status"] == "delivered"
print("late delivered orders, timestamp vs estimated date (old):", (old_late & delivered).sum())
print("late delivered orders, delivery date vs estimated date (new):", (df["is_late"].fillna(False) & delivered).sum())

late delivered orders, timestamp vs estimated date (old): 7826
late delivered orders, delivery date vs estimated date (new): 6534


####  Order Dates and Delivery Metrics

**Investigation:** The order purchase timestamp was used to extract date-related features. Additional metrics were created to measure delivery duration, the duration of intermediate steps, and to compare actual delivery dates with estimated delivery dates.

**Transformation:** The following columns were added:
- `order_year`: Year of the order purchase.
- `order_month`: Month of the order purchase.
- `order_day`: Day of the month when the order was purchased.
- `order_weekday`: Name of the weekday when the order was purchased.
- `delivery_duration_days`: Time elapsed between purchase and customer delivery, measured in days.
- `delivery_delay_days`: Number of **calendar days** between the customer delivery date and the estimated delivery date.
- `is_late`: `True` when `delivery_delay_days > 0`.
- `approval_hours`: Time between purchase and approval, in hours.
- `carrier_to_customer_days`: Time between carrier handover and customer delivery, in days.

**Why calendar days:** `order_estimated_delivery_date` has no time part (it is stored at 00:00). Comparing it with the full delivery timestamp would make an order delivered at 14:00 on the promised day look 0.58 days late. With the timestamp comparison, 7,826 delivered orders were late; with the date comparison, 6,534 are. The difference (1,292 orders) was delivered on the promised day.

**Interpretation of `delivery_delay_days` column:**
- **Positive value:** Delivery was later than estimated.
- **Zero value:** Delivery was made on the estimated date.
- **Negative value:** Delivery was made earlier than estimated.
- **Missing value:** No information is available for comparison.

**Handling Missing Values:** Missing timestamps were preserved. Therefore, delivery metrics remain missing when the required dates are unavailable.

**Validation:** The new columns were inspected to verify their values and missing-value counts.

**Conclusion:** These features provide a foundation for analyzing order trends over time, weekday purchasing patterns, delivery duration, and delivery performance against estimated delivery dates.

### validation after cleaning :

In [119]:
### validation :
df = orders_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate order_id:", df["order_id"].duplicated().sum())
print("Repeated customer_id (informational):", df["customer_id"].duplicated().sum())

# 3. Data types
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
print("\nData types:")
print(df.dtypes)
print("All date columns are datetime:", all(str(df[c].dtype).startswith("datetime") for c in date_cols))

# 4. Order status values
print("\nOrder status:")
print(df["order_status"].value_counts(dropna=False))

# 5. Chronological consistency
checks = {
    "Purchase after approval": df["order_purchase_timestamp"] > df["order_approved_at"],
    "Customer delivery before carrier": df["order_delivered_customer_date"] < df["order_delivered_carrier_date"],
    "Approval after customer delivery": df["order_approved_at"] > df["order_delivered_customer_date"],
}
print("\nRemaining chronological inconsistencies (should be 0):")
for name, cond in checks.items():
    print(f"{name}: {cond.fillna(False).sum()}")
print("Carrier before purchase (kept, flagged):", df["flag_carrier_before_purchase"].sum())
print("Approval after carrier (kept, flagged):", df["flag_approval_after_carrier"].sum())

# 6. Status vs. timestamps (flag only)
delivered = df["order_status"] == "delivered"
print("\nDelivered orders without customer delivery date (flagged):",
      (delivered & df["order_delivered_customer_date"].isna()).sum())
print("Delivered orders without carrier date (flagged):",
      (delivered & df["order_delivered_carrier_date"].isna()).sum())
print("Delivered orders without approval date (flagged):",
      (delivered & df["order_approved_at"].isna()).sum())

# 7. Date range and derived columns
print("\nPurchase date range:", df["order_purchase_timestamp"].min(), "->", df["order_purchase_timestamp"].max())
print("Negative delivery durations:", (df["delivery_duration_days"] < 0).sum())
print("Delivery duration (days):", df["delivery_duration_days"].min().round(2), "-", df["delivery_duration_days"].max().round(2))
print("Delivery delay (calendar days):", df["delivery_delay_days"].min(), "-", df["delivery_delay_days"].max())
print("Late delivered orders:", (df["is_late"].fillna(False) & delivered).sum())
print("Missing delivery metrics:", df["delivery_duration_days"].isna().sum())

# 8. Relationship with other tables
print("\nOrders without matching customer:",
      (~df["customer_id"].isin(customers_clean["customer_id"])).sum())
print("Orders without any item (informational):",
      (~df["order_id"].isin(order_items_clean["order_id"])).sum())

Table shape: (99441, 21)
Missing values per column:
order_approved_at                 221
order_delivered_carrier_date     1806
order_delivered_customer_date    2965
delivery_duration_days           2965
delivery_delay_days              2965
is_late                          2965
approval_hours                    221
carrier_to_customer_days         2989
dtype: int64
Exact duplicate rows: 0
Duplicate order_id: 0
Repeated customer_id (informational): 0

Data types:
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
flag_carrier_before_purchase               bool
flag_customer_before_carrier               bool
flag_approval_after_customer               bool
flag

In [120]:
#save orders_clean (parquet keeps the column types)
orders_clean.to_parquet(PROCESSED + "orders_clean.parquet", index=False)

### Data Cleaning Summary — Orders

**Overview:** The `orders` table (99,441 rows, 8 original columns, 21 after preparation) was inspected and prepared for further analysis. The work focused on missing timestamps, data types, chronological consistency, and the creation of date and delivery features, while preserving all order records.

**Key Activities:**
- **Missing values:** Three timestamp columns had missing values: `order_approved_at` (160, 0.16%), `order_delivered_carrier_date` (1,783, 1.79%), and `order_delivered_customer_date` (2,965, 2.98%). Most belong to orders whose status shows they never reached that stage (e.g., `shipped`, `canceled`, `unavailable`, `invoiced`, `processing`), so they are expected. A small number are inconsistent with the order status: 8 `delivered` orders have no customer delivery date, 2 have no carrier date, and 14 have no approval date. Dropping them would lose valid order data, so no rows were removed and no timestamps were imputed. The inconsistencies were documented.
- **Duplicates:** No exact duplicate rows and no duplicate `order_id` values were found, so `order_id` is a valid primary key.
- **Data types:** All five timestamp columns were stored as strings and were converted to `datetime`. Two malformed values in `order_delivered_customer_date` were corrected before conversion (`2018-02-22 007:55` and `2018-06-18:0 23:07:33`). The conversion introduced no new missing values.
- **Order status:** Eight valid statuses were found, with no unexpected values. `delivered` accounts for 96,478 orders.
- **Timestamp anomalies:** The chronological order of events was checked, and every anomaly was flagged in its own column:
  - 166 carrier dates earlier than the purchase date: **kept and flagged** (`flag_carrier_before_purchase`). The median gap is 0.6 hours, so these look like recording artefacts.
  - 23 carrier dates later than the customer delivery date: carrier date set to `NaT` (`flag_customer_before_carrier`).
  - 61 approval dates later than the customer delivery date: approval date set to `NaT` (`flag_approval_after_customer`).
  - 1,298 approval dates later than the carrier date: kept and flagged (`flag_approval_after_carrier`). These should be excluded only from approval-to-carrier durations.
- **Date coverage:** Purchases range from 2016-09-04 to 2018-10-17.
- **New features:** `order_year`, `order_month`, `order_day`, `order_weekday`, `delivery_duration_days` (purchase to customer delivery), `delivery_delay_days` (calendar days between delivery date and estimated date; positive means late), `is_late`, `approval_hours`, and `carrier_to_customer_days`. Delivery metrics are missing (2,965) wherever the customer delivery date is missing. 6,534 delivered orders are late.

**Data Integrity:** No rows were deleted. Changes were limited to the data type conversions, the two corrected timestamps, and the 84 timestamps set to `NaT`. After cleaning, the missing counts are 221 for `order_approved_at`, 1,806 for `order_delivered_carrier_date`, and 2,965 for `order_delivered_customer_date`.

**Validation:** Final checks covered table dimensions, missing values, exact and `order_id` duplicates, data types, order status values, remaining chronological inconsistencies, consistency between status and timestamps, date range, derived columns, and the relationship with the `customers` and `order_items` tables.

**Conclusion:** The `orders` table is ready for exploratory analysis and for joining with `customers` (through `customer_id`) and with `order_items`, `payments`, and `reviews` (through `order_id`). Delivery-performance analysis should use only orders with a customer delivery date (typically `delivered` status), and analyses of intermediate steps should exclude the rows flagged for that step.

## customers Table :

### missing values

In [121]:
df = customers_clean

#missing overview
missing_overview =pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


 **Decision :** there is no missing value in this table for now so we do nothing

### duplicates :

In [122]:
df = customers_clean

#checking total duplicates
print(f"Exact duplicates: {df.duplicated().sum()}")

#check customer_id duplicates
print(f"customer_id duplicates: {df["customer_id"].duplicated().sum()}")

Exact duplicates: 0
customer_id duplicates: 0


**Decision** : there is no duplicates in whole table and customer_id column.

### Data types :

In [123]:
df = customers_clean

display(df.dtypes)

display(df["customer_zip_code_prefix"].head())

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

0    14409
1     9790
2     1151
3     8775
4    13056
Name: customer_zip_code_prefix, dtype: int64

In [124]:
#convert zip_code_prefix type to string
df["customer_zip_code_prefix"] = df["customer_zip_code_prefix"].astype("string").str.zfill(5)

#validation
print(df["customer_zip_code_prefix"].dtype)

display(df["customer_zip_code_prefix"].head())

string


0    14409
1    09790
2    01151
3    08775
4    13056
Name: customer_zip_code_prefix, dtype: string

#### Data Types conclusion

**Investigation:** The data types of the columns were inspected. The `customer_zip_code_prefix` column was stored as an integer, causing leading zeros in some postal codes to be removed.

**Decision:** The column was converted to string to preserve the original postal code format and treat postal codes as identifiers rather than numerical values.

**Action:** The values were converted to strings and padded with leading zeros to ensure a five-character format.

**Validation:** The column's data type and sample values were checked to confirm that leading zeros were preserved.

### invalid and suspicious values :

In [125]:
df = customers_clean

#check customer_city unique value
print("unique number of cities:",df["customer_city"].nunique())
display(df["customer_city"].value_counts())

#check customer_state unique value
print("\nunique number of states:",df["customer_state"].nunique())
display(df["customer_state"].unique())

unique number of cities: 4119


customer_city
sao paulo              15540
rio de janeiro          6882
belo horizonte          2773
brasilia                2131
curitiba                1521
                       ...  
siriji                     1
natividade da serra        1
monte bonito               1
sao rafael                 1
eugenio de castro          1
Name: count, Length: 4119, dtype: int64


unique number of states: 27


<ArrowStringArray>
['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA', 'MS', 'CE',
 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB', 'TO', 'PI', 'AC', 'SE',
 'RR']
Length: 27, dtype: str

In [126]:
city = df["customer_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())

display(
    df.loc[
        suspicious_city,
        ["customer_id", "customer_city", "customer_state"]
    ].head(10)
)

Suspicious city values: 0


,customer_id,customer_city,customer_state


In [127]:
#Check the relationship with the order table
missing_customers = ~orders_clean["customer_id"].isin(customers_clean["customer_id"])

print("Orders without matching customer:", missing_customers.sum())

Orders without matching customer: 0


#### invalid and suspicious values conclusion

**Investigation:** unique values of customer city and state were checked and examined suspicious city. then we checked relationship of customer_id in orders table with customers table

**Decision:** we found no invalid or suspicious values and everything were normal, and also all customers_id in orders table were matched with customers table .


### validation :

In [128]:
### validation :
df = customers_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate customer_id:", df["customer_id"].duplicated().sum())
print("Repeated customer_unique_id (expected):", df["customer_unique_id"].duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Postal code format
invalid_zip = ~df["customer_zip_code_prefix"].str.fullmatch(r"\d{5}", na=False)
print("\nInvalid postal code prefixes:", invalid_zip.sum())

# 5. City and state checks
city = df["customer_city"]
print("Unique cities:", city.nunique())
print("Suspicious city values (empty or whitespace-padded):",
      (city.isna() | city.str.strip().eq("") | city.ne(city.str.strip())).sum())
print("Cities not fully standardized (uppercase, accents, odd symbols, extra spaces):",
      (~city.str.fullmatch(r"[a-z0-9]+(?:[ '\-][a-z0-9]+)*", na=False)).sum())
print("Unique states:", df["customer_state"].nunique())

# 6. Customer identifiers
print("\nUnique customer_id:", df["customer_id"].nunique())
print("Unique customer_unique_id:", df["customer_unique_id"].nunique())

# 7. Relationship with other tables
print("\nOrders without matching customer:",
      (~orders_clean["customer_id"].isin(df["customer_id"])).sum())
print("Customers without any order (informational):",
      (~df["customer_id"].isin(orders_clean["customer_id"])).sum())
# ZIP coverage in geolocation is checked in the final cross-table section,
# after geolocation ZIP prefixes are converted to the same type

Table shape: (99441, 5)
Total missing values: 0
Exact duplicate rows: 0
Duplicate customer_id: 0
Repeated customer_unique_id (expected): 3345

Data types:
customer_id                    str
customer_unique_id             str
customer_zip_code_prefix    string
customer_city                  str
customer_state                 str
dtype: object

Invalid postal code prefixes: 0
Unique cities: 4119
Suspicious city values (empty or whitespace-padded): 0
Cities not fully standardized (uppercase, accents, odd symbols, extra spaces): 0
Unique states: 27

Unique customer_id: 99441
Unique customer_unique_id: 96096

Orders without matching customer: 0
Customers without any order (informational): 0


In [129]:
#save customers_clean
customers_clean.to_parquet(PROCESSED + "customers_clean.parquet", index=False)

### Data Cleaning Summary — Customers

**Overview:** The `customers` table (99,441 rows, 5 columns) was inspected and prepared for further analysis. The work focused on completeness, uniqueness, data types, and the consistency of city and state values, while preserving all customer records.

**Key Activities:**
- **Missing values:** No missing values were found in any column, so no action was needed.
- **Duplicates:** No exact duplicate rows and no duplicate `customer_id` values were found, so `customer_id` is a valid primary key. `customer_unique_id` repeats across records, which is expected: one real customer can have several `customer_id` values, one per order. The number of unique `customer_unique_id` values (96,096) is therefore the best estimate of the real customer count.
- **Data types:** `customer_zip_code_prefix` was stored as an integer, which removed leading zeros from some postal codes. It was converted to string and padded to five characters, so postal codes are treated as identifiers rather than numbers.
- **City and state values:** 4,119 unique city names were found, with no empty or whitespace-padded values. City names are already lowercase and accent-free (e.g., `sao paulo`), so no standardization was needed, and this is the format used as the reference for the `geolocation` and `sellers` tables. 27 unique state codes were found, matching the number of Brazilian federative units.
- **Relationship with orders:** Every `customer_id` in the `orders` table has a matching record in `customers`.

**Data Integrity:** No rows were deleted. The only modification is the ZIP code type conversion. Typos or non-standard city entries (if any) were not corrected.

**Validation:** Final checks covered table dimensions, missing values, exact and `customer_id` duplicates, data types, postal code format, city and state values, customer identifiers, and the relationship with the `orders` table. ZIP coverage in `geolocation` is checked in the final cross-table section (279 customer records have a prefix not found there).

**Conclusion:** The `customers` table is ready for exploratory analysis and for joining with `orders` through `customer_id` and with `geolocation` through the ZIP code prefix. Customer-level analysis (e.g., repeat purchases, lifetime value) should group by `customer_unique_id`, not `customer_id`. Since `geolocation` can have several coordinates per prefix, it should be aggregated to one location per prefix before joining, to avoid multiplying rows.

## products Table :

### missing values:

In [130]:
df = products_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

missing_overview = missing_overview[missing_overview["missing_count"] > 0].sort_values(by="missing_percent",ascending=False)

display(missing_overview)

,missing_count,missing_percent
product_category_name,610,1.851234
product_name_lenght,610,1.851234
product_description_lenght,610,1.851234
product_photos_qty,610,1.851234
product_weight_g,2,0.006070
product_length_cm,2,0.006070
product_height_cm,2,0.006070
product_width_cm,2,0.006070


In [131]:
#investigation
# Check whether missing values occur in the same rows
missing_columns = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

print("rows where all columns are missing:")
print(df[missing_columns].isna().all(axis=1).sum())

#check physical missing column occur in the same rows
physical_columns = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print(f"\nrows where all physical columns are missing: {df[physical_columns].isna().all(axis=1).sum()}")
display(df[df[physical_columns].isna().all(axis=1)])

#check order_items of product with no information
print("\nnumber of orders with this product that has no information except id :")
display(order_items_clean.loc[order_items_clean["product_id"] == "5eb564652db742ff8f28759cd8d2652a", "order_id"].count())

rows where all columns are missing:
610

rows where all physical columns are missing: 2


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



number of orders with this product that has no information except id :


np.int64(17)

#### conclusion

**Investigation :** we took a summary for missing values, The products dataset contained missing values in both descriptive and physical attributes. but after investigation we figured out each of them had missing in a same row and also a row has no information except `product_id`.

**Decision :** these products are related to orders in **order_items** table, so we couldn't drop rows with missing (even though that column has no information, because we found out 17 orders placed with this product ). This preserves the relationship between products and order items.


### duplicates :

In [132]:
df = products_clean

#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

#check product_id duplicates
print("Duplicate product_id:",df["product_id"].duplicated().sum())

Exact duplicates: 0
Duplicate product_id: 0


**conclusion :** there is no duplicates in whole table and product_id column

### Data types :

In [133]:
df = products_clean

display(df.dtypes)

product_id                        str
product_category_name             str
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

**conclusion :** all data types are fine

### invalid values :

In [134]:
#check values range

numerical_cols = ["product_name_lenght",
                  "product_description_lenght",
                  "product_photos_qty",
                  "product_weight_g" ,
                  "product_length_cm",
                  "product_height_cm",
                  "product_width_cm"]

display(df[numerical_cols].agg(["min", "max"]))

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
min,5.0,4.0,1.0,0.0,7.0,2.0,6.0
max,76.0,3992.0,20.0,40425.0,105.0,105.0,118.0


**conclusion :** all of them are in proper range

### prepration

In [135]:
# Merge the English names onto products
products_clean = products_clean.merge(
    product_name_translation,
    on="product_category_name",
    how="left"
)

# Add the two missing translations manually
manual = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_and_food_preparers",
}
products_clean["product_category_name"] = (
    products_clean["product_category_name_english"]
    .fillna(products_clean["product_category_name"].map(manual))
)

# Replace the Portuguese column with the English one
products_clean = products_clean.drop(columns="product_category_name_english")

# Products without a category form their own "unknown" group
products_clean["product_category_name"] = products_clean["product_category_name"].fillna("unknown")

# Fix the misspelled column names from the source file
products_clean = products_clean.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

df = products_clean
display(df["product_category_name"].value_counts())

product_category_name
bed_bath_table               3029
sports_leisure               2867
furniture_decor              2657
health_beauty                2444
housewares                   2335
                             ... 
fashion_childrens_clothes       5
home_comfort_2                  5
pc_gamer                        3
security_and_services           2
cds_dvds_musicals               1
Name: count, Length: 74, dtype: int64

#### Result :

The `product_category_name` column in the `products` table was in Portuguese, so it was translated to English using the `product_category_name_translation` table.

The row count is unchanged (32,951), so no duplicates were introduced.

Two categories (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`) have no entry in the translation table, so their English column is missing (13 products). They were added manually.

The 610 products without a category were assigned to an `"unknown"` category, so category-level analysis keeps them as their own group. Their other descriptive columns stay missing.

The misspelled source columns `product_name_lenght` and `product_description_lenght` were renamed to `product_name_length` and `product_description_length`.

### validation:

In [136]:
### validation :
df = products_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate product_id:", df["product_id"].duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Category translation
print("\nUnique categories (including 'unknown'):", df["product_category_name"].nunique())
print("Products in the 'unknown' category:", (df["product_category_name"] == "unknown").sum())
print("Sample categories:", df["product_category_name"].drop_duplicates().head(5).tolist())

# 5. Value ranges
numerical_cols = ["product_name_length", "product_description_length", "product_photos_qty",
                  "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
print("\nValue ranges:")
print(df[numerical_cols].agg(["min", "max"]).T)
print("Negative values:", (df[numerical_cols] < 0).sum().sum())
print("Zero weight (flagged):", (df["product_weight_g"] == 0).sum())

# 6. Referential integrity with order_items
print("\nOrphan product_id in order_items (not in products):",
      (~order_items_clean["product_id"].isin(df["product_id"])).sum())
print("Products never ordered (informational):",
      (~df["product_id"].isin(order_items_clean["product_id"])).sum())

Table shape: (32951, 9)
Missing values per column:
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
Exact duplicate rows: 0
Duplicate product_id: 0

Data types:
product_id                        str
product_category_name             str
product_name_length           float64
product_description_length    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

Unique categories (including 'unknown'): 74
Products in the 'unknown' category: 610
Sample categories: ['perfumery', 'art', 'sports_leisure', 'baby', 'housewares']

Value ranges:
                            min      max
product_name_length         5.0     76.0
product_description

In [137]:
#save products_clean
products_clean.to_parquet(PROCESSED + "products_clean.parquet", index=False)

### Data Cleaning Summary — Products

**Overview:** The `products` table (32,951 rows, 9 columns) was inspected and prepared for further analysis. The work focused on completeness, uniqueness, data types, the plausibility of numerical attributes, and translating category names, while preserving all product records.

**Key Activities:**
- **Missing values:** 610 products (1.85%) have missing values in `product_category_name`, `product_name_lenght`, `product_description_lenght`, and `product_photos_qty`, always in the same rows. In addition, 2 products have missing values in all four physical measurement columns (`product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm`). One of them has no information other than its `product_id` and is referenced by 17 rows in `order_items`. Because these products are linked to orders and no reliable replacement values exist, the numeric missing values were retained.
- **Duplicates:** No exact duplicate rows and no duplicate `product_id` values were found, so `product_id` is a valid primary key.
- **Data types:** All columns already had appropriate types, so no conversion was needed.
- **Value ranges:** All numerical attributes are within plausible ranges, with no negative values. The minimum `product_weight_g` is 0, which is physically implausible. These records were kept and flagged (4 products), and should be excluded from weight-based analysis such as freight cost per gram.
- **Category translation:** `product_category_name` was in Portuguese and was translated to English using the `product_category_name_translation` table. Two categories missing from that table (`pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`, 13 products) were translated manually. The 610 products without a category were assigned to `"unknown"`, giving 74 categories in total (73 real ones plus `"unknown"`). The row count stayed at 32,951, so the merge introduced no duplicates.
- **Column names:** `product_name_lenght` and `product_description_lenght` were renamed to `product_name_length` and `product_description_length`.

**Data Integrity:** No rows were deleted. The modifications are the category translation, the `"unknown"` category, and the two column renames. Numeric missing values and zero weights were retained and documented.

**Validation:** Final checks covered table dimensions, missing values, exact and `product_id` duplicates, data types, category translation (on the translated table), value ranges, and referential integrity with the `order_items` table.

**Conclusion:** The `products` table is ready for exploratory analysis and for joining with `order_items` through `product_id`. Analyses using physical attributes should account for the 2 products with missing measurements and the zero-weight records.

## order_items Table :

### missing values :

In [138]:
#missing overview

df = order_items_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


**there is no missing values in the table**

### duplicates :

In [139]:
df = order_items_clean

#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

#check ids duplicates
print("Duplicate order_id:",df["order_id"].duplicated().sum())
print("Duplicate product_id:",df["product_id"].duplicated().sum())
print("Duplicate seller_id:",df["seller_id"].duplicated().sum())

Exact duplicates: 0
Duplicate order_id: 13984
Duplicate product_id: 79699
Duplicate seller_id: 109555


**conclusion :** we don't have any duplicates row in total but we figured out that the id columns are not unique and all of them reffered to other tables

### data types :

In [140]:
df = order_items_clean

display(df.dtypes)

order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

In [141]:
#convert limit_date type to datetime
df["shipping_limit_date"] = pd.to_datetime(df["shipping_limit_date"],
                                           format="%Y-%m-%d %H:%M:%S")

# Validate data types
display(df.dtypes)

# Check missing values after conversion
display(df["shipping_limit_date"].isna().sum())

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

np.int64(0)

**conclusion :** `shipping_limit_date` type were converted to datetime and proper format.

**validation :** after validation and checking missing values after converstion no missing values or wrong data types were observed

### invalid and suspicious values :

In [142]:
#check ranges

df = order_items_clean

#check cost ranges
price_cols = ["price","freight_value"]

display(df[price_cols].agg(["min", "max"]).T)

#check shipping_limit_date range
display(df["shipping_limit_date"].agg(["min", "max"]))

,min,max
price,0.85,6735.00
freight_value,0.00,409.68


min   2016-09-19 00:15:34
max   2020-04-09 22:35:08
Name: shipping_limit_date, dtype: datetime64[us]

In [143]:
#check zero freight_value
df = order_items_clean

zero_freight = df["freight_value"].eq(0)

print("Zero freight rows:", zero_freight.sum())

zero_freight = df[df["freight_value"] == 0]

print("Unique orders:", zero_freight["order_id"].nunique())
print("Unique products:", zero_freight["product_id"].nunique())
print("Unique sellers:", zero_freight["seller_id"].nunique())

#check relationship between zero shipping costs with sellers and products
display(zero_freight["product_id"].value_counts().head())
display(zero_freight["seller_id"].value_counts().head())


Zero freight rows: 383
Unique orders: 339
Unique products: 10
Unique sellers: 9


product_id
53b36df67ebb7c41585e8d54d6772e08    187
aca2eb7d00ea1a7b8ebd4e68314663af     98
422879e10f46682990de24d770e7f83d     56
7a10781637204d8d10485c71a6108a2e     27
f1c7f353075ce59d8a6f3cf58f419c9c      9
Name: count, dtype: int64

seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
Name: count, dtype: int64

**conclusion :** A total of 383 order items have zero freight value, belonging to 339 unique orders, 10 unique products, and 9 sellers. These records are highly concentrated among a small number of sellers and products. No negative freight values were found. This distribution suggests that zero freight values may reflect specific shipping arrangements rather than random data errors. Therefore, these records were retained without modification.

### validation

In [144]:
### validation :
df = order_items_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate (order_id, order_item_id) pairs:",
      df.duplicated(subset=["order_id", "order_item_id"]).sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Value ranges
print("\nInvalid prices (<= 0):", (df["price"] <= 0).sum())
print("Negative freight values:", (df["freight_value"] < 0).sum())
print("Zero freight values (kept):", (df["freight_value"] == 0).sum())

# 5. Referential integrity with related tables
print("\nOrphan order_id (not in orders):",
      (~df["order_id"].isin(orders_clean["order_id"])).sum())
print("Orphan product_id (not in products):",
      (~df["product_id"].isin(products_clean["product_id"])).sum())
print("Orphan seller_id (not in sellers):",
      (~df["seller_id"].isin(sellers_clean["seller_id"])).sum())

# 6. Shipping limit date beyond the order period (flag only)
last_order_date = orders_clean["order_purchase_timestamp"].max()
late_limit = df["shipping_limit_date"] > last_order_date + pd.Timedelta(days=90)
print("\nShipping limit date far beyond the order period (flagged):", late_limit.sum())

Table shape: (112650, 7)
Total missing values: 0
Exact duplicate rows: 0
Duplicate (order_id, order_item_id) pairs: 0

Data types:
order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

Invalid prices (<= 0): 0
Negative freight values: 0
Zero freight values (kept): 383

Orphan order_id (not in orders): 0
Orphan product_id (not in products): 0
Orphan seller_id (not in sellers): 0

Shipping limit date far beyond the order period (flagged): 4


In [145]:
# save order_items_clean
order_items_clean.to_parquet(PROCESSED + "order_items_clean.parquet", index=False)

### Data Cleaning Summary — Order Items

**Overview:** The `order_items` table (112,650 rows, 7 columns) was inspected and prepared for further analysis. The work focused on completeness, uniqueness, data types, and the plausibility of price, freight, and date values, while preserving all item records.

**Key Activities:**
- **Missing values:** No missing values were found in any column, so no action was needed.
- **Duplicates:** No exact duplicate rows were found. `order_id`, `product_id`, and `seller_id` repeat many times, which is expected: each is a foreign key, and one order can contain several items. The table has no single unique column, but an item is identified by the combination of `order_id` and `order_item_id`.
- **Data types:** `shipping_limit_date` was stored as a string and was converted to `datetime`. The conversion introduced no missing values. All other columns already had appropriate types.
- **Value ranges:** `price` ranges from 0.85 to 6,735.00, with no zero or negative prices. `freight_value` ranges from 0.00 to 409.68, with no negative values.
- **Zero freight values:** 383 items (339 orders, 10 products, 9 sellers) have a freight value of 0. They are concentrated in a few sellers and products, which suggests specific shipping arrangements (e.g., free shipping) rather than random errors. These records were retained unchanged.

**Data Integrity:** No rows were deleted and no values were modified, apart from the data type conversion. Relationships with the `orders`, `products`, and `sellers` tables were checked as part of the final validation.

**Validation:** Final checks covered table dimensions, missing values, exact duplicates, uniqueness of (`order_id`, `order_item_id`), data types, price and freight ranges, referential integrity with related tables, and flagged shipping limit dates.

**Conclusion:** The `order_items` table is ready for exploratory analysis and for joining with `orders`, `products`, and `sellers`. The remaining limitations are the zero freight values and the 4 shipping limit dates far beyond the order period (up to 2020-04-09), both retained and documented rather than altered.

## geolocation table :

### missing values :

In [146]:
df = geolocation_clean

#missing overview
missing_overview =pd.DataFrame({
    "missing_count" : df.isna().sum(),
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)


,missing_count,missing_percent
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


no missing values were found

### data types

In [147]:
df = geolocation_clean

display(df.dtypes)
display(df["geolocation_zip_code_prefix"].head())

geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

0    1037
1    1046
2    1046
3    1041
4    1035
Name: geolocation_zip_code_prefix, dtype: int64

In [148]:
#convert zip_code_prefix type to string
df["geolocation_zip_code_prefix"] = df["geolocation_zip_code_prefix"].astype("string").str.zfill(5)

#validation
print(df["geolocation_zip_code_prefix"].dtype)

display(df["geolocation_zip_code_prefix"].head())

string


0    01037
1    01046
2    01046
3    01041
4    01035
Name: geolocation_zip_code_prefix, dtype: string

#### Data Types conclusion

**Investigation:** The data types of the columns were inspected. The `goelocation_zip_code_prefix` column was stored as an integer, causing leading zeros in some postal codes to be removed.

**Decision:** The column was converted to string to preserve the original postal code format and treat postal codes as identifiers rather than numerical values.

**Action:** The values were converted to strings and padded with leading zeros to ensure a five-character format.

**Validation:** The column's data type and sample values were checked to confirm that leading zeros were preserved.

### duplicates :

In [149]:
df = geolocation_clean

#checking total duplicates
print(f"Exact duplicates: {df.duplicated().sum()}")

#check customer_id duplicates
print(f"zip-code prefix duplicates: {df["geolocation_zip_code_prefix"].duplicated().sum()}")

Exact duplicates: 261831
zip-code prefix duplicates: 981148


In [150]:
print("Before drop:", geolocation_clean.shape)

geolocation_clean = geolocation_clean.drop_duplicates()

print("After drop:", geolocation_clean.shape)
print("Exact duplicates:",geolocation_clean.duplicated().sum())

Before drop: (1000163, 5)
After drop: (738332, 5)
Exact duplicates: 0


#### Conclusion

The duplicate analysis identified 261,831 exact duplicate rows. These records were removed because they contained no additional information. However, duplicate ZIP code prefixes were retained, as a single postal code prefix can correspond to multiple geographic coordinates.

**Final decision:** Exact duplicate rows were removed, while repeated ZIP code prefixes were preserved to maintain the geographical information associated with each postal code.

### invalid & suspicious values :

In [151]:
df = geolocation_clean

#check geolocation_city unique value
print("unique number of cities:",df["geolocation_city"].nunique())
display(df["geolocation_city"].value_counts())

#check geolocation_state unique value
print("\nunique number of states:",df["geolocation_state"].nunique())
display(df["geolocation_state"].unique())

unique number of cities: 8011


geolocation_city
sao paulo                     79927
rio de janeiro                35177
são paulo                     19718
belo horizonte                19474
curitiba                      11263
                              ...  
almirante tamandaré do sul        1
três palmeiras                    1
são valentim                      1
três arroios                      1
ciríaco                           1
Name: count, Length: 8011, dtype: int64


unique number of states: 27


<ArrowStringArray>
['SP', 'RN', 'AC', 'RJ', 'ES', 'MG', 'BA', 'SE', 'PE', 'AL', 'PB', 'CE', 'PI',
 'MA', 'PA', 'AP', 'AM', 'RR', 'DF', 'GO', 'RO', 'TO', 'MT', 'MS', 'RS', 'PR',
 'SC']
Length: 27, dtype: str

In [152]:
city = df["geolocation_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())

display(
    df.loc[suspicious_city].head(10)
)

Suspicious city values: 1


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
670078,40243,-12.987814,-38.505742,salvador,BA


In [153]:
#fix suspicious city
df["geolocation_city"] = df["geolocation_city"].str.strip()

#validation
city = df["geolocation_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())


Suspicious city values: 0


In [154]:
#Standardization of cities
df = geolocation_clean

before = df["geolocation_city"].nunique()

df["geolocation_city"] = (
    df["geolocation_city"]
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")                  # split letters from accents (ã -> a + ~)
    .str.encode("ascii", errors="ignore")   # drop the accent marks
    .str.decode("ascii")
    .str.replace(r"[^a-z0-9\s'\-]", " ", regex=True)  # remove odd symbols, keep ' and -
    .str.replace(r"\s+", " ", regex=True)   # collapse repeated spaces
    .str.strip()
)

after = df["geolocation_city"].nunique()
print(f"Unique cities: {before} -> {after}")

# Standardizing can create new exact duplicates, so remove them
print("Duplicates created:", df.duplicated().sum())
geolocation_clean = df.drop_duplicates().reset_index(drop=True)
print("Final shape:", geolocation_clean.shape)

Unique cities: 8010 -> 5962
Duplicates created: 17838
Final shape: (720494, 5)


### coordinate outliers :

In [155]:
#points outside Brazil's approximate bounds
df = geolocation_clean

out_of_brazil = ~(
    df["geolocation_lat"].between(-33.75, 5.27)
    & df["geolocation_lng"].between(-73.99, -34.79)
)
print("Coordinates outside Brazil bounds:", out_of_brazil.sum())
display(df.loc[out_of_brazil, "geolocation_state"].value_counts().head())

#remove them, since they would pull the location of their ZIP prefix far away
geolocation_clean = df.loc[~out_of_brazil].reset_index(drop=True)
print("Shape after removing them:", geolocation_clean.shape)

Coordinates outside Brazil bounds: 33


geolocation_state
PE    6
RJ    5
PA    5
BA    4
RS    3
Name: count, dtype: int64

Shape after removing them: (720461, 5)


#### conclusion

Points outside Brazil's approximate bounds cannot be correct for a Brazilian ZIP prefix. They were removed, because they would distort the location computed for their prefix.

#### conclusion

The `geolocation_city` column was standardized to improve data consistency and reduce variations in city names.

The cleaning process included:
- Removing leading and trailing whitespace.
- Converting all city names to lowercase.
- Normalizing Unicode characters and removing accent marks.
- Replacing consecutive whitespace characters with a single space.

After standardization, the number of unique city names was compared with the original count to evaluate the changes.

Finally, duplicate records were identified and removed to maintain data consistency. The cleaned dataset was saved as `geolocation_clean`.

### validation

In [156]:
### validation :
df = geolocation_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Postal code format
invalid_zip = ~df["geolocation_zip_code_prefix"].str.fullmatch(r"\d{5}", na=False)
print("\nInvalid postal code prefixes:", invalid_zip.sum())

# 5. City standardization
city = df["geolocation_city"]
print("\nUnique cities:", city.nunique())
print("Empty or missing city values:", (city.isna() | city.eq("")).sum())
print("Cities not fully standardized (uppercase, accents, odd symbols, extra spaces):",
      (~city.str.fullmatch(r"[a-z0-9]+(?:[ '\-][a-z0-9]+)*", na=False)).sum())

# 6. State checks
print("Unique states:", df["geolocation_state"].nunique())

# 7. Relationship with other tables
print("Customer ZIP prefixes not found in geolocation (informational):",
      (~customers_clean["customer_zip_code_prefix"].isin(geolocation_clean["geolocation_zip_code_prefix"])).sum())

# 8. Coordinates within Brazil's approximate bounds (flag only)
out_of_brazil = ~(
    df["geolocation_lat"].between(-33.75, 5.27)
    & df["geolocation_lng"].between(-73.99, -34.79)
)
print("Coordinates outside Brazil bounds (should be 0):", out_of_brazil.sum())


Table shape: (720461, 5)
Total missing values: 0
Exact duplicate rows: 0

Data types:
geolocation_zip_code_prefix     string
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Invalid postal code prefixes: 0

Unique cities: 5958
Empty or missing city values: 0
Cities not fully standardized (uppercase, accents, odd symbols, extra spaces): 5
Unique states: 27
Customer ZIP prefixes not found in geolocation (informational): 279
Coordinates outside Brazil bounds (should be 0): 0


In [157]:
# save geolocation_clean
geolocation_clean.to_parquet(PROCESSED + "geolocation_clean.parquet", index=False)

### one location per ZIP prefix :

In [158]:
#one row per ZIP prefix, so other tables can join without multiplying rows
#median is used instead of mean, because it is not pulled by a few distant points
df = geolocation_clean

geolocation_prefix = (
    df.groupby("geolocation_zip_code_prefix")
    .agg(
        lat=("geolocation_lat", "median"),
        lng=("geolocation_lng", "median"),
        city=("geolocation_city", lambda s: s.mode().iloc[0]),
        state=("geolocation_state", lambda s: s.mode().iloc[0]),
        n_points=("geolocation_lat", "size"),
        n_states=("geolocation_state", "nunique"),
    )
    .reset_index()
)

#how far the mean would have been from the median
mean_coords = df.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()
shift = np.hypot(
    mean_coords["geolocation_lat"].values - geolocation_prefix["lat"].values,
    mean_coords["geolocation_lng"].values - geolocation_prefix["lng"].values,
)

print("Prefixes:", len(geolocation_prefix))
print("Duplicate prefixes (should be 0):", geolocation_prefix["geolocation_zip_code_prefix"].duplicated().sum())
print("Prefixes where mean and median differ by more than 0.5 degrees:", (shift > 0.5).sum())
print("Prefixes mapped to more than one state:", (geolocation_prefix["n_states"] > 1).sum())
display(geolocation_prefix[geolocation_prefix["n_states"] > 1])

geolocation_prefix.to_parquet(PROCESSED + "geolocation_prefix.parquet", index=False)

Prefixes: 19010
Duplicate prefixes (should be 0): 0
Prefixes where mean and median differ by more than 0.5 degrees: 85
Prefixes mapped to more than one state: 8


,geolocation_zip_code_prefix,lat,lng,city,state,n_points,n_states
367,02116,-23.519402,-46.584880,sao paulo,SP,10,2
1668,04011,-23.579522,-46.644503,sao paulo,SP,62,2
6505,21550,-22.862002,-43.353996,rio de janeiro,RJ,145,2
6695,23056,-22.919086,-43.610982,rio de janeiro,RJ,31,2
14662,72915,-15.723643,-48.268492,aguas lindas de goias,GO,9,2
15876,78557,-11.877390,-55.501693,sinop,MT,76,2
16143,79750,-22.248897,-53.346529,nova andradina,MS,152,2
16253,80630,-25.469776,-49.274464,curitiba,PR,77,2


### Data Cleaning Summary — Geolocation

**Overview:** The `geolocation` table (1,000,163 rows, 5 columns) was inspected and prepared for further analysis. The work focused on completeness, duplicates, data types, the consistency of city and state values, and the reliability of coordinates.

**Key Activities:**
- **Missing values:** No missing values were found in any column, so no action was needed.
- **Data types:** `geolocation_zip_code_prefix` was stored as an integer, which removed leading zeros from some postal codes. It was converted to string and padded to five characters, so postal codes are treated as identifiers rather than numbers.
- **Duplicates (first pass):** 261,831 exact duplicate rows were found and removed because they carried no additional information, leaving 738,332 rows. Repeated ZIP code prefixes were kept, since one prefix can map to several coordinates.
- **City standardization:** City names contained whitespace issues, accent variants, and case variants (e.g., `são paulo` and `sao paulo`). They were standardized by trimming whitespace, converting to lowercase, removing accents, replacing special characters, and collapsing repeated spaces. This matches the format already used in the `customers` table. The number of unique cities decreased from 8,011 to 5,962.
- **Duplicates (second pass):** Standardization created 17,838 additional exact duplicates, which were removed, leaving 720,494 rows. The index was then reset.
- **Coordinate outliers:** 33 points outside Brazil's approximate bounds were removed, leaving 720,461 rows.
- **State values:** 27 unique state codes were found, which matches the number of Brazilian federative units. No invalid values were observed.
- **One location per prefix:** a separate table, `geolocation_prefix` (19,010 rows), holds one row per ZIP prefix with the **median** latitude and longitude, the most frequent city and state, the number of points, and the number of states. The median was chosen because in 85 prefixes the mean is more than 0.5° (about 55 km) away from it. 8 prefixes map to more than one state (kept, see `n_states`).

**Data Integrity:** Only exact duplicate rows and points outside Brazil were removed. Apart from the ZIP code type conversion and city name standardization, no values were modified. Typos and non-accent spelling errors in city names were not corrected.

**Validation:** Final checks covered table dimensions, missing values, exact duplicates, data types, postal code format, city name standardization, the number of states, coordinate bounds, and the uniqueness of `geolocation_prefix`.

**Conclusion:** Other tables should join to `geolocation_prefix` (not to `geolocation_clean`) through the ZIP code prefix, which keeps one row per customer or seller. 279 customer records and 7 sellers have a prefix not found in `geolocation`, so they will have missing coordinates.

## sellers table :

### missing values :

In [159]:
df = sellers_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
seller_id,0,0.0
seller_zip_code_prefix,0,0.0
seller_city,0,0.0
seller_state,0,0.0


No missing values were found in the sellers.

### duplicates :

In [160]:
# Check for duplicated sellers
print("Exact duplicate rows:", df.duplicated().sum())
print(f"Number of Duplicated Sellers: {df["seller_id"].duplicated().sum()}")

Exact duplicate rows: 0
Number of Duplicated Sellers: 0


There are no duplicated sellers in the dataset.
Everything seems to be fine!

### data types :

In [161]:
df = sellers_clean

display(df.dtypes)

seller_id                   str
seller_zip_code_prefix    int64
seller_city                 str
seller_state                str
dtype: object

In [162]:
#convert zip_code_prefix type to string
df["seller_zip_code_prefix"] = df["seller_zip_code_prefix"].astype("string").str.zfill(5)

#validation
print(df["seller_zip_code_prefix"].dtype)

display(df["seller_zip_code_prefix"].head())

string


0    13023
1    13844
2    20031
3    04195
4    12914
Name: seller_zip_code_prefix, dtype: string

### invalid & suspicious values :

In [163]:
df = sellers_clean

#check seller_city unique value
print("unique number of cities:",df["seller_city"].nunique())
display(df["seller_city"].value_counts())

#check seller_state unique value
print("\nunique number of states:",df["seller_state"].nunique())
display(df["seller_state"].unique())

unique number of cities: 611


seller_city
sao paulo                 694
curitiba                  127
rio de janeiro             96
belo horizonte             68
ribeirao preto             52
                         ... 
aparecida de goiania        1
bandeirantes                1
vitoria de santo antao      1
palotina                    1
leme                        1
Name: count, Length: 611, dtype: int64


unique number of states: 23


<ArrowStringArray>
['SP', 'RJ', 'PE', 'PR', 'GO', 'SC', 'BA', 'DF', 'RS', 'MG', 'RN', 'MT', 'CE',
 'PB', 'AC', 'ES', 'RO', 'PI', 'MS', 'SE', 'MA', 'AM', 'PA']
Length: 23, dtype: str

In [164]:
#check Suspicious city
city = df["seller_city"]

suspicious_city = (
    city.isna()
    | city.str.strip().eq("")
    | city.ne(city.str.strip())
)

print("Suspicious city values:", suspicious_city.sum())

display(
    df.loc[suspicious_city].head(10)
)

Suspicious city values: 0


,seller_id,seller_zip_code_prefix,seller_city,seller_state


In [165]:
#Standardization of cities
df = sellers_clean

before = df["seller_city"].nunique()

df["seller_city"] = (
    df["seller_city"]
    .str.strip()
    .str.lower()
    .str.normalize("NFKD")                  # split letters from accents (ã -> a + ~)
    .str.encode("ascii", errors="ignore")   # drop the accent marks
    .str.decode("ascii")
    .str.replace(r"[^a-z0-9\s'\-]", " ", regex=True)  # remove odd symbols, keep ' and -
    .str.replace(r"\s+", " ", regex=True)   # collapse repeated spaces
    .str.strip()
)

after = df["seller_city"].nunique()
print(f"Unique cities: {before} -> {after}")

# Standardizing can create new exact duplicates, so remove them
print("Duplicates created:", df.duplicated().sum())
sellers_clean = df.drop_duplicates().reset_index(drop=True)
print("Final shape:", sellers_clean.shape)

Unique cities: 611 -> 605
Duplicates created: 0
Final shape: (3095, 4)


### validation

In [166]:
### validation :
df = sellers_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Total missing values:", df.isna().sum().sum())
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate seller_id:", df["seller_id"].duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Postal code format
invalid_zip = ~df["seller_zip_code_prefix"].str.fullmatch(r"\d{5}", na=False)
print("\nInvalid postal code prefixes:", invalid_zip.sum())

# 5. City standardization
city = df["seller_city"]
print("\nUnique cities:", city.nunique())
print("Empty or missing city values:", (city.isna() | city.eq("")).sum())
print("Cities not fully standardized (uppercase, accents, odd symbols, extra spaces):",
      (~city.str.fullmatch(r"[a-z0-9]+(?:[ '\-][a-z0-9]+)*", na=False)).sum())

# 6. State checks
print("Unique states:", df["seller_state"].nunique())

# 7. Relationship with other tables
print("\nSellers not found in order_items (informational):",
      (~df["seller_id"].isin(order_items_clean["seller_id"])).sum())
print("Seller ZIP prefixes not found in geolocation (informational):",
      (~df["seller_zip_code_prefix"].isin(geolocation_clean["geolocation_zip_code_prefix"])).sum())

Table shape: (3095, 4)
Total missing values: 0
Exact duplicate rows: 0
Duplicate seller_id: 0

Data types:
seller_id                    str
seller_zip_code_prefix    string
seller_city                  str
seller_state                 str
dtype: object

Invalid postal code prefixes: 0

Unique cities: 605
Empty or missing city values: 0
Cities not fully standardized (uppercase, accents, odd symbols, extra spaces): 4
Unique states: 23

Sellers not found in order_items (informational): 0
Seller ZIP prefixes not found in geolocation (informational): 7


In [167]:
# save sellers_clean
sellers_clean.to_parquet(PROCESSED + "sellers_clean.parquet", index=False)

### Data Cleaning Summary — Sellers

**Overview:** The `sellers` table (3,095 rows, 4 columns) was inspected and prepared for further analysis. The work focused on completeness, uniqueness, data types, and the consistency of city and state values.

**Key Activities:**
- **Missing values:** No missing values were found in any column, so no action was needed.
- **Duplicates:** No exact duplicate rows and no duplicate `seller_id` values were found, so `seller_id` is a valid primary key.
- **Data types:** `seller_zip_code_prefix` was stored as an integer, which removed leading zeros from some postal codes. It was converted to string and padded to five characters, so postal codes are treated as identifiers rather than numbers.
- **City standardization:** No empty or whitespace-padded city values were found. City names were still standardized by trimming whitespace, converting to lowercase, removing accents, replacing special characters, and collapsing repeated spaces. This matches the format used in the `customers` and `geolocation` tables. The number of unique cities decreased from 611 to 605. The step created no new duplicate rows.
- **State values:** 23 unique state codes were found, all valid Brazilian state codes. Sellers are present in 23 of the 27 federative units.

**Data Integrity:** No rows were deleted. The only modified values are the ZIP code type conversion and the city name standardization. Typos and non-standard city entries (if any) were not corrected.

**Validation:** Final checks covered table dimensions, missing values, exact and `seller_id` duplicates, data types, postal code format, city name standardization, the number of states, and the relationship with the `order_items` and `geolocation` tables.

**Conclusion:** The `sellers` table is ready for exploratory analysis and for joining with `order_items` (through `seller_id`) and `geolocation` (through the ZIP code prefix). Since `geolocation` can have several coordinates per prefix, it should be aggregated to one location per prefix before joining, to avoid multiplying rows.

## payments table :

### missing values :

In [168]:
df = payments_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

display(missing_overview)

,missing_count,missing_percent
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


No missing values were found!

### duplicates :

In [169]:
# Duplicate rows
print("Exact duplicates:", df.duplicated().sum())

# Duplicate payment identifiers
print(f"Duplicate payment records: {df.duplicated(subset=["order_id", "payment_sequential"]).sum()}")

Exact duplicates: 0
Duplicate payment records: 0


No duplicated rows or payments were found. Repeated order_id values are normal because one order can have multiple payment records.

### data types

In [170]:
df = payments_clean
display(df.dtypes)

order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int64
payment_value           float64
dtype: object

The data types seem to be well-defined.

### invalid & suspicious values :

In [171]:
#check payment method unique
df["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [172]:
#check not defiend values
display(df[df["payment_type"] == "not_defined"])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0


In [173]:
#check order status of not defiend payment types
df[
    df["payment_type"] == "not_defined"
].merge(
    orders_clean[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0,canceled
1,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0,canceled
2,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0,canceled


#### Investigation of Undefined Payment Types

Three records were found with `payment_type = 'not_defined'`. All three records had a `payment_value` of 0.0 and were associated with canceled orders.

This pattern suggests that these records may represent orders canceled before a payment method was recorded. Since there is no reliable evidence to determine the actual payment method, these records were retained without modification.

**Decision:** The three records were kept, and their original values were preserved.

In [174]:
#check installment & value range
display(df[["payment_installments" , "payment_value"]].agg(["min", "max"]).T)

,min,max
payment_installments,0.0,24.00
payment_value,0.0,13664.08


In [175]:
#check zero payment_installments
display(df[df["payment_installments"] == 0])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [176]:
#check all orders with this ids
target_orders = [
    "744bade1fcf9ff3f31d860ace076d422",
    "1a57108394169c0b47d8f876acc9ba2d"
]

display(df[df["order_id"].isin(target_orders)].sort_values(by=["order_id", "payment_sequential"]))

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


In [177]:
df = payments_clean
#nullable integer type, so the column stays integer with missing values
df["payment_installments"] = df["payment_installments"].astype("Int64")
df.loc[df["payment_installments"] == 0, "payment_installments"] = pd.NA

#validation
display(df[df["order_id"].isin(target_orders)])

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,<NA>,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,<NA>,129.94


#### conclusion

Two records were found with `payment_installments = 0`. Both were credit card payments with positive payment values.

Further investigation showed that each order had only one payment record in the dataset, and both records had `payment_sequential = 2`. Since no additional payment records were available, the actual installment counts could not be determined. A `payment_sequential = 2` without a sequence 1 also means that the first payment record of these orders is **missing** from the dataset.

**Decision:** The two records were retained, but their zero installment values were treated as missing because zero is not a valid installment count. The column uses the nullable integer type (`Int64`), so it stays integer.

In [178]:
#check zero payments(not defiend didn't consider)
zero_payments = ((df["payment_value"] == 0) & (df["payment_type"] != "not_defined"))

display(zero_payments.sum())
display(df.loc[zero_payments])

np.int64(6)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [179]:
df.loc[zero_payments].merge(
    orders_clean[["order_id", "order_status"]],
    on="order_id",
    how="left"
)

,order_id,payment_sequential,payment_type,payment_installments,payment_value,order_status
0,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0,delivered
1,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0,shipped
2,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0,delivered
3,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0,delivered
4,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0,shipped
5,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0,delivered


#### conclusion

Six records were identified with `payment_value = 0` and a defined payment type. All six records used the `voucher` payment method.

Further investigation showed that these records belonged to five orders: four with a `delivered` status and one with a `shipped` status. All six records had `payment_installments = 1`.

The payment history of these orders revealed that they also contained other payment records, most of which had positive payment values. This suggests that the zero-value voucher records may be part of multiple-payment transactions rather than necessarily indicating data errors.

**Decision:** All six records were retained without modification. Their zero payment values were preserved because there was insufficient evidence to classify them as invalid or determine alternative values.

In [180]:
#Checking if the sum of prices in an order matches its payment value:

payment_totals = df.groupby("order_id")["payment_value"].sum()

item_totals = (
    order_items_clean.assign(total= order_items_clean["price"] + order_items_clean["freight_value"])
    .groupby("order_id")["total"].sum()
)

comparison = pd.concat(
    [payment_totals, item_totals],
    axis=1,
    keys=["payment_total", "item_total"]
)

#rounded to cents, so floating-point noise does not decide the 0.01 threshold
comparison["difference"] = (
    comparison["payment_total"] - comparison["item_total"]
).round(2)

mismatches = comparison[
    comparison["difference"].abs() > 0.01
].copy()

print("Payment > Item total:",
      (mismatches["difference"] > 0.01).sum())

print("Payment < Item total:",
      (mismatches["difference"] < -0.01).sum())

print("Total mismatches:", len(mismatches))
mismatches["abs_difference"] = (
    mismatches["difference"].abs()
)

display(
    mismatches["abs_difference"].describe()
)

Payment > Item total: 264
Payment < Item total: 39
Total mismatches: 303


count    303.000000
mean      10.789505
std       17.322488
min        0.020000
25%        2.200000
50%        5.900000
75%       12.850000
max      182.810000
Name: abs_difference, dtype: float64

In [181]:
#most difference between total price and payment valu
display(comparison.loc[comparison["difference"].abs().nlargest(10).index])

,payment_total,item_total,difference
order_id,,,
ce6d150fb29ada17d2082f4847107665,1586.47,1403.66,182.81
6e5fe7366a2e1bfbf3257dba0af1267f,406.92,287.91,119.01
70b742795bc441e94a44a084b6d9ce7a,578.82,466.93,111.89
996c7e73600ad3723e8627ab7bef81e4,664.43,587.90,76.53
70b7e94ea46d3e8b5bc12a50186edaf0,274.84,213.15,61.69
bc2c82b0ef78d2252b6176d1972db7c9,303.02,242.01,61.01
af9ffff2ce6b3defd34fd4c78857a379,466.97,413.17,53.80
262118ce178bb3e4590a3adcf6d62e6b,126.12,177.74,-51.62
bfdb5bbb06458d600a33d61f5f287472,394.36,348.93,45.43


In [182]:
#are the mismatches related to installments (credit card interest)?
installments = df.groupby("order_id")["payment_installments"].max()
comparison["multi_installment"] = installments.reindex(comparison.index) > 1

matched = comparison["difference"].abs() <= 0.01
higher = comparison["difference"] > 0.01

print("Share of orders with more than 1 installment:")
print(f"  all orders:                  {comparison['multi_installment'].mean():.1%}")
print(f"  orders without mismatch:     {comparison.loc[matched, 'multi_installment'].mean():.1%}")
print(f"  orders with mismatch:        {comparison.loc[~matched & comparison['difference'].notna(), 'multi_installment'].mean():.1%}")
print(f"  orders paying more than items: {comparison.loc[higher, 'multi_installment'].mean():.1%}")

Share of orders with more than 1 installment:
  all orders:                  51.5%
  orders without mismatch:     51.4%
  orders with mismatch:        88.8%
  orders paying more than items: 94.3%


#### conclusion:

A consistency check was performed by comparing the total payment value of each order with the sum of its item prices and freight values. Differences were rounded to cents first, so 1-cent rounding noise is not counted.

The analysis identified **303 orders** with a difference greater than 0.01 between these two totals:

- **264 orders (87.1%)** had a payment total higher than the item total.
- **39 orders (12.9%)** had a payment total lower than the item total.

The absolute differences ranged from 0.02 to 182.81, with a mean difference of 10.79 and a median difference of 5.90.

**Likely cause:** 88.8% of the mismatching orders were paid in more than one installment, compared with 51.5% of all orders. Among orders where the customer paid more than the item total, the share is 94.3%. Credit card **installment interest** therefore explains most of the differences.

**Decision:** All records were retained without modification. For analysis:
- **GMV and seller revenue** come from `order_items` (`price` + `freight_value`).
- **The amount the customer actually paid** comes from `payments`.

### validation:

In [183]:
### validation :
df = payments_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values and duplicates
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate (order_id, payment_sequential) pairs:",
      df.duplicated(subset=["order_id", "payment_sequential"]).sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Payment types
print("\nPayment types:")
print(df["payment_type"].value_counts(dropna=False))

# 5. Value ranges
print("\nInstallments range:", df["payment_installments"].min(), "-", df["payment_installments"].max())
print("Zero installments (should be 0):", (df["payment_installments"] == 0).sum())
print("Negative payment values:", (df["payment_value"] < 0).sum())
print("Zero payment values (kept):", (df["payment_value"] == 0).sum())
print("not_defined payment type (kept):", (df["payment_type"] == "not_defined").sum())

# 6. Referential integrity with related tables
print("\nOrphan order_id (not in orders):",
      (~df["order_id"].isin(orders_clean["order_id"])).sum())
print("Orders without any payment record (informational):",
      (~orders_clean["order_id"].isin(df["order_id"])).sum())

# 7. Payment total vs item total per order (flag only)
payment_totals = df.groupby("order_id")["payment_value"].sum()
item_totals = (
    order_items_clean.assign(total=order_items_clean["price"] + order_items_clean["freight_value"])
    .groupby("order_id")["total"].sum()
)
diff = (payment_totals - item_totals).dropna().round(2)
print("\nOrders where payment total differs from item total (flagged):",
      (diff.abs() > 0.01).sum())

Table shape: (103886, 5)
Missing values per column:
payment_installments    2
dtype: int64
Exact duplicate rows: 0
Duplicate (order_id, payment_sequential) pairs: 0

Data types:
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      Int64
payment_value           float64
dtype: object

Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Installments range: 1 - 24
Zero installments (should be 0): 0
Negative payment values: 0
Zero payment values (kept): 9
not_defined payment type (kept): 3

Orphan order_id (not in orders): 0
Orders without any payment record (informational): 1

Orders where payment total differs from item total (flagged): 303


In [184]:
# save payments_clean
payments_clean.to_parquet(PROCESSED + "payments_clean.parquet", index=False)

### Data Cleaning Summary — Payments

**Overview:** The `payments` table (103,886 rows, 5 columns) was inspected and prepared for further analysis. The work focused on completeness, duplicates, data types, and the plausibility of payment types, installments, and payment values, while preserving all payment records.

**Key Activities:**
- **Missing values:** No missing values were found in any column.
- **Duplicates:** No exact duplicate rows and no duplicate (`order_id`, `payment_sequential`) pairs were found. Repeated `order_id` values are expected, because one order can have several payment records.
- **Data types:** All columns already had appropriate types. `payment_installments` was later changed to nullable `Int64` to hold the 2 missing values.
- **Payment types:** Five values were found: `credit_card` (76,795), `boleto` (19,784), `voucher` (5,775), `debit_card` (1,529), and `not_defined` (3).
- **Undefined payment type:** The 3 `not_defined` records all have a payment value of 0 and belong to canceled orders, which suggests the orders were canceled before a payment method was recorded. They were retained unchanged.
- **Zero installments:** 2 credit card records had `payment_installments = 0`, which is not a valid installment count. Each belongs to an order with only one payment record (with `payment_sequential = 2`, so the first payment record is missing), and the real value could not be determined. The records were kept and the zero values were set to missing.
- **Zero payment values:** 6 voucher records (5 orders: 4 delivered, 1 shipped) have a payment value of 0. These orders also contain other payment records, so the zeros may be part of multiple-payment transactions rather than errors. They were retained unchanged.
- **Payment vs. item totals:** 303 orders differ by more than 0.01 after rounding to cents: 264 have a higher payment total and 39 a lower one (median difference 5.90, maximum 182.81). 88.8% of these orders used installments (51.5% overall), so installment interest is the likely cause. No values were changed.

**Data Integrity:** No rows were deleted. The only modification is converting the 2 zero installment values to missing. Zero payment values, `not_defined` records, and total mismatches were retained and documented.

**Validation:** Final checks covered table dimensions, missing values, exact and (`order_id`, `payment_sequential`) duplicates, data types, payment types, installment and payment value ranges, referential integrity with `orders`, and the comparison of payment totals with item totals.

**Conclusion:** The `payments` table is ready for exploratory analysis and for joining with `orders` through `order_id`. Because an order can have several payment records, the table should be aggregated to one row per order (e.g., the sum of `payment_value`) before joining, to avoid multiplying rows. Use `order_items` for GMV and seller revenue, and `payments` for the amount customers paid.

## reviews table:

### missing values:

In [185]:
#Missing Values overview

df = reviews_clean

missing_overview = pd.DataFrame({
    "missing_count" : df.isna().sum() ,
    "missing_percent" : df.isna().mean() * 100
})

missing_overview = missing_overview[missing_overview["missing_count"] > 0].sort_values(by="missing_percent",ascending=False)
display(missing_overview)

,missing_count,missing_percent
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532


#### conclusion

The missing-value analysis identified missing values in two columns:

- `review_comment_title`: 87,656 missing values (88.34%).
- `review_comment_message`: 58,247 missing values (58.70%).

These columns contain optional customer feedback. Customers can submit a review score without providing a written title or comment. Therefore, missing values in these columns do not necessarily indicate data quality issues.

**Decision:** Missing values were retained as `NaN` rather than replaced with empty strings or artificial text, since they represent the absence of written feedback rather than invalid data.

### duplicates:

In [186]:
#checking total duplicates
print("Exact duplicates:", df.duplicated().sum())

Exact duplicates: 0


In [187]:
#checking review_id duplicates
display(df["review_id"].duplicated().sum())

np.int64(814)

In [188]:
#Check if records with the same review_id are exactly the same or have different information:
duplicates = df[
    df["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicates.groupby("review_id").nunique().head()

,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
review_id,,,,,,
00130cbe1f9d422698c812ed8ded1919,2,1,0,1,1,1
0115633a9c298b6a98bcbe4eee75345f,2,1,0,0,1,1
0174caf0ee5964646040cd94e15ac95e,2,1,0,1,1,1
017808d29fd1f942d97e50184dfb4c13,2,1,0,0,1,1
0254bd905dc677a6078990aad3331a36,2,1,0,1,1,1


In [189]:
#Number of records per Review ID
review_id_counts = (
    df.groupby("review_id")
    .size()
    .loc[lambda x: x > 1]
)

print("Duplicated review IDs:", len(review_id_counts))
print("Total records with duplicated IDs:", review_id_counts.sum())
print("Maximum occurrences:", review_id_counts.max())

Duplicated review IDs: 789
Total records with duplicated IDs: 1603
Maximum occurrences: 3


In [190]:
#check which columns are actually different, except order_id.
duplicates = df[
    df["review_id"].duplicated(keep=False)
]

cols = [
    col for col in df.columns
    if col not in ["review_id", "order_id"]
]

variation = (
    duplicates.groupby("review_id")[cols]
    .nunique()
)

display(variation.max().sort_values(ascending=False))

review_score               1
review_comment_title       1
review_comment_message     1
review_creation_date       1
review_answer_timestamp    1
dtype: int64

In [191]:
#Does each order_id have only one review_id or is it itself linked to multiple review IDs?
reviews_clean.groupby("order_id")["review_id"].nunique().value_counts().sort_index()

review_id
1    98126
2      543
3        4
Name: count, dtype: int64

In [192]:
#who are the orders that share a review_id?
shared = (
    reviews_clean[reviews_clean["review_id"].duplicated(keep=False)]
    .merge(orders_clean[["order_id", "customer_id", "order_purchase_timestamp"]], on="order_id")
    .merge(customers_clean[["customer_id", "customer_unique_id"]], on="customer_id")
)
per_review = shared.groupby("review_id").agg(
    customers=("customer_unique_id", "nunique"),
    purchase_times=("order_purchase_timestamp", "nunique"),
)

print("Duplicated review IDs:", len(per_review))
print("Linked to a single customer_unique_id:", f"{(per_review['customers'] == 1).mean():.1%}")
print("All linked orders purchased at the same moment:", f"{(per_review['purchase_times'] == 1).mean():.1%}")

Duplicated review IDs: 789
Linked to a single customer_unique_id: 100.0%
All linked orders purchased at the same moment: 32.2%


#### Conclusion

The duplicate analysis identified **789 duplicated `review_id` values**, representing **1,603 records**. Each duplicated review ID appeared in either two or three records.

Within each group of duplicated `review_id` values, all columns except `order_id` contained identical values: the same review is linked to several orders.

The relationship between orders and reviews was also examined:

- **98,126 orders** were associated with one review ID.
- **543 orders** were associated with two different review IDs.
- **4 orders** were associated with three different review IDs.

**Explanation:** in **100%** of the duplicated review IDs, all linked orders belong to the **same customer** (`customer_unique_id`), and in 32.2% of them the orders were even purchased at the same moment. One customer wrote one review, and Olist attached it to several of that customer's orders. This is a property of how the data was recorded, not an error.

**Decision:** No records were removed or modified. Rules for analysis:
- **Order-level analysis:** when an order has several reviews, use the **latest** one (highest `review_answer_timestamp`), which reflects the customer's final experience.
- **Review-level statistics** (e.g., score distribution): deduplicate on `review_id` first, otherwise shared reviews are counted 2–3 times.

### data types:

In [193]:
df = reviews_clean
display(df.dtypes)

review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object

In [194]:
# Convert date columns to datetime

date_cols = [
    "review_creation_date",
    "review_answer_timestamp"
]

for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="%Y-%m-%d %H:%M:%S"
    )

# Validate data types
display(df.dtypes)

# Check missing values after conversion
display(df[date_cols].isna().sum())

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

review_creation_date       0
review_answer_timestamp    0
dtype: int64

date columns converted to datetime

### invalid & suspicious values:

In [195]:
#check rating range
display(df["review_score"].agg(["min", "max"]).T)

min    1
max    5
Name: review_score, dtype: int64

the rating range is correct

In [196]:
#Timestamp consistency
answer_before_creation = df["review_answer_timestamp"] < df["review_creation_date"]
print("Answer timestamp before creation date (flagged):", answer_before_creation.sum())

Answer timestamp before creation date (flagged): 0


### validation:

In [197]:
### validation :
df = reviews_clean

# 1. Table dimensions
print("Table shape:", df.shape)

# 2. Missing values (optional comment fields) and duplicates
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate (review_id, order_id) pairs:",
      df.duplicated(subset=["review_id", "order_id"]).sum())
print("Duplicated review_id values (kept, flagged):", df["review_id"].duplicated().sum())

# 3. Data types
print("\nData types:")
print(df.dtypes)

# 4. Value ranges
print("\nReview score range:", df["review_score"].min(), "-", df["review_score"].max())
print("Invalid scores (not 1-5):", (~df["review_score"].between(1, 5)).sum())
print("Date range:", df["review_creation_date"].min(), "->", df["review_creation_date"].max())

# 5. Timestamp consistency (flag only)
answer_before_creation = df["review_answer_timestamp"] < df["review_creation_date"]
print("Answer timestamp before creation date (flagged):", answer_before_creation.sum())

# 6. Order-review relationship
reviews_per_order = df.groupby("order_id")["review_id"].nunique()
print("\nOrders with more than one review_id (flagged):", (reviews_per_order > 1).sum())

# 7. Referential integrity with orders
print("Orphan order_id (not in orders):",
      (~df["order_id"].isin(orders_clean["order_id"])).sum())
print("Orders without any review (informational):",
      (~orders_clean["order_id"].isin(df["order_id"])).sum())

Table shape: (99224, 7)
Missing values per column:
review_comment_title      87656
review_comment_message    58247
dtype: int64
Exact duplicate rows: 0
Duplicate (review_id, order_id) pairs: 0
Duplicated review_id values (kept, flagged): 814

Data types:
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

Review score range: 1 - 5
Invalid scores (not 1-5): 0
Date range: 2016-10-02 00:00:00 -> 2018-08-31 00:00:00
Answer timestamp before creation date (flagged): 0

Orders with more than one review_id (flagged): 547
Orphan order_id (not in orders): 0
Orders without any review (informational): 768


In [198]:
# save reviews_clean
reviews_clean.to_parquet(PROCESSED + "reviews_clean.parquet", index=False)

### Data Cleaning Summary — Reviews

**Overview:** The `reviews` table (99,224 rows, 7 columns) was inspected and prepared for further analysis. The work focused on completeness, duplicates, data types, and the plausibility of review scores, while preserving all review records.

**Key Activities:**
- **Missing values:** `review_comment_title` has 87,656 missing values (88.34%) and `review_comment_message` has 58,247 (58.70%). These fields are optional, because a customer can leave a score without writing a comment, so the missing values do not indicate a data quality problem. They were kept as `NaN` rather than replaced with empty strings or artificial text.
- **Duplicates (exact rows):** No exact duplicate rows were found.
- **Duplicates (`review_id`):** 789 `review_id` values appear more than once, covering 1,603 records, with each ID appearing two or three times. Within each group, all columns except `order_id` are identical. In every case the linked orders belong to the same customer: one review was attached to several orders of that customer.
- **Order-review relationship:** 98,126 orders have one review ID, 543 have two, and 4 have three. The relationship between orders and reviews is therefore not strictly one-to-one, and `review_id` is not a unique identifier.
- **Data types:** `review_creation_date` and `review_answer_timestamp` were stored as strings and were converted to `datetime`. The conversion introduced no missing values. All other columns already had appropriate types. Note that `review_creation_date` is the date the satisfaction survey was **sent** to the customer, not the date the customer answered it.
- **Value ranges:** `review_score` ranges from 1 to 5, as expected.
- **Coverage:** review dates end on 2018-08-31, while orders go to 2018-10-17.

**Data Integrity:** No rows were deleted and no values were modified, apart from the data type conversion. Missing comments, duplicated `review_id` values, and orders with multiple reviews were retained and documented.

**Validation:** Final checks covered table dimensions, missing values, exact and `review_id` duplicates, data types, score range, timestamp consistency, the number of reviews per order, and referential integrity with the `orders` table.

**Conclusion:** The `reviews` table is ready for exploratory analysis and for joining with `orders` through `order_id`. Before joining, keep the **latest** review per order to avoid multiplying order rows. Review-level statistics should deduplicate on `review_id`. Text-based analysis should account for the high share of missing comments.

## joins validation

In [199]:
def check_merge(left, right, on, name, expected, sum_col=None, how="left"):
    """Merge, then report row counts, multiplication, unmatched keys and (optionally) a numeric total from the left table."""
    n_before = len(left)
    merged = left.merge(right, on=on, how=how, indicator=True)
    n_after = len(merged)
    unmatched = (merged["_merge"] == "left_only").sum()
    right_dup_keys = right[on].duplicated().sum()

    print(f"--- {name} | key: {on} | expected: {expected}")
    print(f"rows before: {n_before:,} -> after: {n_after:,} | "
          f"{'OK, no multiplication' if n_after == n_before else f'ROWS CHANGED (+{n_after - n_before:,})'}")
    print(f"duplicate keys in right table: {right_dup_keys:,} | left rows without a match: {unmatched:,}")
    if sum_col:
        b, a = left[sum_col].sum(), merged[sum_col].sum()
        print(f"sum of {sum_col}: {b:,.2f} -> {a:,.2f} | {'OK' if abs(a - b) < 0.01 else f'INFLATED by {a - b:,.2f}'}")
    print()
    return merged.drop(columns="_merge")

In [200]:
# Pre-aggregated versions (one row per join key)
payments_agg = payments_clean.groupby("order_id", as_index=False).agg(payment_total=("payment_value", "sum"))
# latest review per order (the final experience of the customer)
reviews_agg  = (reviews_clean.sort_values("review_answer_timestamp")
                .drop_duplicates("order_id", keep="last")[["order_id", "review_score"]])
geo_agg      = geolocation_prefix  # one row per prefix, median coordinates

print("Safe joins: many-to-one, rows must NOT change\n")
_ = check_merge(orders_clean, customers_clean, "customer_id", "orders -> customers", "m:1")
_ = check_merge(order_items_clean, orders_clean[["order_id", "order_status"]], "order_id", "order_items -> orders", "m:1", sum_col="price")
_ = check_merge(order_items_clean, products_clean, "product_id", "order_items -> products", "m:1", sum_col="price")
_ = check_merge(order_items_clean, sellers_clean, "seller_id", "order_items -> sellers", "m:1", sum_col="price")

print("Risky joins done WITHOUT aggregation: expect multiplication\n")
_ = check_merge(orders_clean, payments_clean, "order_id", "orders -> payments (raw)", "1:m")
_ = check_merge(orders_clean, reviews_clean, "order_id", "orders -> reviews (raw)", "1:m (some orders have 2-3 reviews)")
_ = check_merge(order_items_clean, payments_clean, "order_id", "order_items -> payments (raw)", "m:m, DO NOT USE", sum_col="price")

print("Same joins AFTER aggregation: rows must be unchanged\n")
_ = check_merge(orders_clean, payments_agg, "order_id", "orders -> payments (aggregated)", "1:1")
_ = check_merge(orders_clean, reviews_agg, "order_id", "orders -> reviews (latest review)", "1:1")

# check_merge uses one key name for both sides, so the prefix column is renamed
geo_for_customers = geo_agg.rename(columns={"geolocation_zip_code_prefix": "customer_zip_code_prefix"})
_ = check_merge(customers_clean, geo_for_customers, "customer_zip_code_prefix", "customers -> geolocation_prefix", "m:1")

# 4) Geolocation: check the aggregated version, and estimate the raw blow-up without running it
geo_counts = geolocation_clean["geolocation_zip_code_prefix"].value_counts()
raw_rows = customers_clean["customer_zip_code_prefix"].map(geo_counts).fillna(1).sum()
print(f"customers -> geolocation (raw) would produce ~{int(raw_rows):,} rows from {len(customers):,}\n")


Safe joins: many-to-one, rows must NOT change

--- orders -> customers | key: customer_id | expected: m:1
rows before: 99,441 -> after: 99,441 | OK, no multiplication
duplicate keys in right table: 0 | left rows without a match: 0

--- order_items -> orders | key: order_id | expected: m:1
rows before: 112,650 -> after: 112,650 | OK, no multiplication
duplicate keys in right table: 0 | left rows without a match: 0
sum of price: 13,591,643.70 -> 13,591,643.70 | OK

--- order_items -> products | key: product_id | expected: m:1
rows before: 112,650 -> after: 112,650 | OK, no multiplication
duplicate keys in right table: 0 | left rows without a match: 0
sum of price: 13,591,643.70 -> 13,591,643.70 | OK

--- order_items -> sellers | key: seller_id | expected: m:1
rows before: 112,650 -> after: 112,650 | OK, no multiplication
duplicate keys in right table: 0 | left rows without a match: 0
sum of price: 13,591,643.70 -> 13,591,643.70 | OK

Risky joins done WITHOUT aggregation: expect multiplic

#### Join Validation Summary

**Purpose:** Before analysis, the main joins between the cleaned tables were checked for unexpected row multiplication, unmatched keys, and distortion of numerical totals.

##### Join keys and expected cardinality

| Join | Key | Cardinality | Rows before → after | Result |
|---|---|---|---|---|
| orders → customers | `customer_id` | many-to-one | 99,441 → 99,441 | Safe |
| order_items → orders | `order_id` | many-to-one | 112,650 → 112,650 | Safe |
| order_items → products | `product_id` | many-to-one | 112,650 → 112,650 | Safe |
| order_items → sellers | `seller_id` | many-to-one | 112,650 → 112,650 | Safe |
| orders → payments (raw) | `order_id` | one-to-many | 99,441 → 103,887 (+4,446) | Multiplies rows |
| orders → reviews (raw) | `order_id` | one-to-many | 99,441 → 99,992 (+551) | Multiplies rows |
| order_items → payments (raw) | `order_id` | many-to-many | 112,650 → 117,604 (+4,954) | Inflates totals |
| orders → payments (aggregated) | `order_id` | one-to-one | 99,441 → 99,441 | Safe |
| orders → reviews (latest review) | `order_id` | one-to-one | 99,441 → 99,441 | Safe |
| customers → geolocation_prefix | ZIP prefix | many-to-one | 99,441 → 99,441 | Safe |
| customers → geolocation (raw) | ZIP prefix | many-to-many | 99,441 → ~10,075,969 (estimated) | Never use raw |

##### Key findings

- **Safe joins:** The four many-to-one joins (`orders → customers`, and `order_items → orders / products / sellers`) did not change the row count. There were no duplicate keys in the right tables and no unmatched rows. The sum of `price` stayed at 13,591,643.70 after each join.
- **Row multiplication in raw joins:** Joining `payments` directly to `orders` added 4,446 rows, because one order can have several payment records. Joining `reviews` directly added 551 rows, because some orders have more than one review.
- **Inflated totals:** Joining `order_items` directly to `payments` raised the total `price` from 13,591,643.70 to 14,209,250.31, an inflation of 617,606.61 (about 4.5%). Each item row is repeated once per payment record of its order, so this join must not be used for revenue analysis.
- **Geolocation:** A raw join on ZIP prefix would turn 99,441 customers into about 10 million rows. Joining `geolocation_prefix` (one row per prefix) keeps the row count.
- **Fix confirmed:** After aggregating `payments` (sum of `payment_value`) and keeping the latest review per order, both joins keep the row count at 99,441.

##### Unmatched keys (expected, not errors)

- 1 order has no payment record.
- 768 orders have no review. These orders will have missing review scores after joining.
- 3 order items belong to orders with no payment record (seen in the raw `order_items → payments` join).
- 279 customer records have a ZIP prefix not found in `geolocation`, so they will have missing coordinates.

##### Rules for the analysis stage

1. Join `orders`, `customers`, `order_items`, `products`, and `sellers` directly. These joins are safe.
2. Aggregate `payments` (sum per order) and `reviews` (latest per order) before joining them to `orders`.
3. Join `customers` and `sellers` to `geolocation_prefix`, never to `geolocation_clean`.
4. Never join `order_items` directly to `payments`. Revenue should come from `order_items` (`price` + `freight_value`) or from the aggregated payment totals, not from both at once.
5. Use `customer_unique_id`, not `customer_id`, for customer-level analysis.

##### Conclusion

All safe joins preserve row counts and totals. The risky joins were identified, their effect was measured, and the aggregation step that prevents row multiplication was verified. The tables are ready to be combined for exploratory analysis.

## final cross-table checks

In [201]:
#run after every table is cleaned, so all keys have their final types
print("Orders without matching customer:",
      (~orders_clean["customer_id"].isin(customers_clean["customer_id"])).sum())
print("Orders without any item (informational):",
      (~orders_clean["order_id"].isin(order_items_clean["order_id"])).sum())
print("Orders without any payment (informational):",
      (~orders_clean["order_id"].isin(payments_clean["order_id"])).sum())
print("Orders without any review (informational):",
      (~orders_clean["order_id"].isin(reviews_clean["order_id"])).sum())
print("Orphan order_id in order_items / payments / reviews:",
      (~order_items_clean["order_id"].isin(orders_clean["order_id"])).sum(),
      (~payments_clean["order_id"].isin(orders_clean["order_id"])).sum(),
      (~reviews_clean["order_id"].isin(orders_clean["order_id"])).sum())
print("Orphan product_id / seller_id in order_items:",
      (~order_items_clean["product_id"].isin(products_clean["product_id"])).sum(),
      (~order_items_clean["seller_id"].isin(sellers_clean["seller_id"])).sum())

geo_prefixes = geolocation_prefix["geolocation_zip_code_prefix"]
print("\nCustomer records with ZIP prefix not in geolocation:",
      (~customers_clean["customer_zip_code_prefix"].isin(geo_prefixes)).sum())
print("Seller records with ZIP prefix not in geolocation:",
      (~sellers_clean["seller_zip_code_prefix"].isin(geo_prefixes)).sum())

Orders without matching customer: 0
Orders without any item (informational): 775
Orders without any payment (informational): 1
Orders without any review (informational): 768
Orphan order_id in order_items / payments / reviews: 0 0 0
Orphan product_id / seller_id in order_items: 0 0

Customer records with ZIP prefix not in geolocation: 279
Seller records with ZIP prefix not in geolocation: 7


## parquet round-trip check

In [202]:
#reload every saved file and make sure the column types survived
saved = {
    "orders_clean": orders_clean,
    "customers_clean": customers_clean,
    "products_clean": products_clean,
    "order_items_clean": order_items_clean,
    "geolocation_clean": geolocation_clean,
    "geolocation_prefix": geolocation_prefix,
    "sellers_clean": sellers_clean,
    "payments_clean": payments_clean,
    "reviews_clean": reviews_clean,
}

for name, frame in saved.items():
    reloaded = pd.read_parquet(PROCESSED + f"{name}.parquet")
    assert reloaded.shape == frame.shape, name
    assert (reloaded.dtypes == frame.dtypes).all(), name
    print(f"{name}: {reloaded.shape} OK")

#ZIP prefixes keep their leading zeros
for name, col in [("customers_clean", "customer_zip_code_prefix"),
                  ("sellers_clean", "seller_zip_code_prefix"),
                  ("geolocation_prefix", "geolocation_zip_code_prefix")]:
    zips = pd.read_parquet(PROCESSED + f"{name}.parquet")[col]
    assert zips.str.fullmatch(r"\d{5}").all(), name
    print(f"{name}.{col}: {zips.dtype}, starts with 0: {zips.str.startswith('0').sum():,} values")

orders_clean: (99441, 21) OK
customers_clean: (99441, 5) OK
products_clean: (32951, 9) OK
order_items_clean: (112650, 7) OK
geolocation_clean: (720461, 5) OK
geolocation_prefix: (19010, 7) OK
sellers_clean: (3095, 4) OK
payments_clean: (103886, 5) OK
reviews_clean: (99224, 7) OK
customers_clean.customer_zip_code_prefix: string, starts with 0: 23,995 values
sellers_clean.seller_zip_code_prefix: string, starts with 0: 1,027 values
geolocation_prefix.geolocation_zip_code_prefix: string, starts with 0: 4,354 values


## Data Cleaning Report: Olist E-commerce Dataset

### 1. Objective and Scope

This report summarizes the cleaning and preparation of the eight Olist tables: `orders`, `customers`, `products`, `order_items`, `geolocation`, `sellers`, `payments`, and `reviews`. The goal was to produce reliable, analysis-ready tables, document every data quality issue found, and make sure tables can be joined without distorting row counts or totals.

The `product_category_name_translation` table was used only as a lookup to translate product categories.

Raw files are in `Data/Raw/`. Cleaned tables are saved to `Data/Processed/` as **Parquet** files (`<table>_clean.parquet`), plus one extra table, `geolocation_prefix.parquet`. Parquet stores the column types inside the file, so ZIP prefixes stay zero-padded strings, timestamps stay `datetime`, and flags stay boolean when the files are reloaded. A round-trip check at the end of the notebook confirms this.

### 2. Cleaning Principles

Each table went through the same workflow: **missing values → duplicates → data types → invalid/suspicious values → preparation → validation → save**. Decisions followed these rules:

1. **Preserve data by default.** Rows were deleted only when they were exact duplicates carrying no extra information, or coordinates outside Brazil.
2. **Fix only what is provably wrong.** Values that were impossible (e.g., a customer delivery before the carrier handover) were set to missing rather than guessed.
3. **Flag, don't alter, when the cause is unclear.** Ambiguous cases were kept unchanged and documented (zero payments, total mismatches, zero freight, timestamps that look like recording artefacts, and similar). Timestamp anomalies have their own `flag_*` columns.
4. **No imputation.** No missing value was filled with an estimate, because no reliable basis for one existed. (Missing product categories were labeled `"unknown"`, which is a label, not an estimate.)
5. **Every step has a recorded investigation, decision, action, and validation.**

### 3. Overview

| Table | Rows before | Rows after | Columns (before → after) | Main actions |
|---|---:|---:|---|---|
| `orders` | 99,441 | 99,441 | 8 → 21 | Datetime conversion, 2 malformed dates fixed, 84 impossible timestamps set to `NaT`, 4 anomaly flags and 9 derived features added |
| `customers` | 99,441 | 99,441 | 5 → 5 | ZIP prefix converted to 5-character string |
| `products` | 32,951 | 32,951 | 9 → 9 | Category names translated to English (2 manually), missing categories set to `"unknown"`, 2 misspelled columns renamed |
| `order_items` | 112,650 | 112,650 | 7 → 7 | `shipping_limit_date` converted to datetime |
| `geolocation` | 1,000,163 | 720,461 | 5 → 5 | ZIP prefix to string, city standardization, 279,669 exact duplicates and 33 out-of-Brazil points removed |
| `geolocation_prefix` (new) | — | 19,010 | 7 | One row per ZIP prefix with median coordinates |
| `sellers` | 3,095 | 3,095 | 4 → 4 | ZIP prefix to string, city standardization |
| `payments` | 103,886 | 103,886 | 5 → 5 | 2 invalid zero installment values set to missing (`Int64`) |
| `reviews` | 99,224 | 99,224 | 7 → 7 | Date columns converted to datetime |

Only `geolocation` lost rows. Everything else was preserved.

### 4. Results by Table

#### 4.1 Orders

- **Missing values:** `order_approved_at` (160), `order_delivered_carrier_date` (1,783), and `order_delivered_customer_date` (2,965). Most belong to orders that never reached that stage (`shipped`, `canceled`, `unavailable`, `invoiced`, `processing`), so they are expected. A few are inconsistent with the status: 8 `delivered` orders had no customer delivery date, 2 had no carrier date, and 14 had no approval date. These were kept and documented.
- **Duplicates:** none. `order_id` is a valid primary key.
- **Data types:** all five timestamp columns were strings and were converted to `datetime`. Two malformed values were corrected first (`2018-02-22 007:55` and `2018-06-18:0 23:07:33`).
- **Order status:** eight valid statuses, no unexpected values. `delivered` accounts for 96,478 orders.
- **Chronological anomalies:** each case has its own flag column:

| Issue | Count | Action | Flag column |
|---|---:|---|---|
| Carrier date before purchase | 166 | Kept (median gap 0.6 hours, likely a recording artefact) | `flag_carrier_before_purchase` |
| Customer delivery before carrier date | 23 | `order_delivered_carrier_date` → `NaT` | `flag_customer_before_carrier` |
| Approval after customer delivery | 61 | `order_approved_at` → `NaT` | `flag_approval_after_customer` |
| Approval after carrier date | 1,298 | Kept (possible recording delay) | `flag_approval_after_carrier` |

- **New features:** `order_year`, `order_month`, `order_day`, `order_weekday`, `delivery_duration_days` (purchase to customer delivery), `delivery_delay_days` (calendar days between the delivery date and the estimated date; positive means late), `is_late`, `approval_hours`, and `carrier_to_customer_days`. Delivery metrics are missing for the 2,965 orders without a customer delivery date.
- **Late deliveries:** 6,534 delivered orders are late. The delay is measured in calendar days, because the estimated date has no time part. A timestamp comparison would have counted 7,826, wrongly including 1,292 orders delivered on the promised day.
- **After cleaning:** missing counts are 221 (`order_approved_at`), 1,806 (`order_delivered_carrier_date`), and 2,965 (`order_delivered_customer_date`). Among `delivered` orders, 8 lack a customer delivery date, 25 lack a carrier date, and 75 lack an approval date (the increase comes from the timestamps set to `NaT`).
- **Date coverage:** purchases from 2016-09-04 to 2018-10-17. Delivery duration ranges from 0.53 to 209.63 days, with no negative values.

#### 4.2 Customers

- No missing values, no duplicate rows, and no duplicate `customer_id` (valid primary key).
- `customer_unique_id` repeats (3,345 repeated values), which is expected: one real customer gets a new `customer_id` per order. There are **99,441 `customer_id` values but 96,096 unique customers**.
- `customer_zip_code_prefix` was converted from integer to a zero-padded 5-character string so leading zeros are preserved.
- 4,119 distinct cities, already lowercase and accent-free (this format was used as the reference for the other tables). 27 states, matching the 27 Brazilian federative units.
- Every order has a matching customer.
- **Geolocation coverage:** 279 customer records (about 0.28%) have a ZIP prefix that does not exist in the cleaned `geolocation` table. They were kept, and will have missing coordinates after joining to `geolocation_prefix`.

#### 4.3 Products

- **Missing values:** 610 products (1.85%) lack category, name length, description length, and photo count, always in the same rows. 2 products also lack all four physical measurements, and one of them has no information except its ID and is referenced by 17 rows in `order_items`. Since these products are tied to orders and there is no basis for imputation, they were kept.
- No duplicates (`product_id` is a valid primary key). Data types were appropriate. All numeric ranges are plausible with no negatives, except **4 products with weight 0**, which is physically implausible; they were kept and flagged.
- **Category translation:** Portuguese names were translated to English via the translation table. `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` (13 products) were missing from that table and were translated manually. The 610 products without a category were assigned to `"unknown"`, giving 74 categories (73 real ones plus `"unknown"`). The row count is unchanged.
- The misspelled source columns `product_name_lenght` and `product_description_lenght` were renamed to `product_name_length` and `product_description_length`.

#### 4.4 Order Items

- No missing values, no exact duplicates. An item is identified by (`order_id`, `order_item_id`), which has no duplicates. Repeated `order_id`, `product_id`, and `seller_id` are expected foreign keys.
- `shipping_limit_date` was converted to `datetime` with no new missing values.
- `price` ranges from 0.85 to 6,735.00 (no zero or negative prices). `freight_value` ranges from 0.00 to 409.68.
- **Zero freight:** 383 items (339 orders, 10 products, 9 sellers). The concentration in a few sellers and products points to specific shipping arrangements (e.g., free shipping) rather than random errors, so they were kept.
- **Date range caveat:** `shipping_limit_date` runs from 2016-09-19 to 2020-04-09. 4 values fall more than 90 days after the last order. They were not modified and should be used with caution.
- All `order_id`, `product_id`, and `seller_id` values exist in their parent tables (no orphans).

#### 4.5 Geolocation

- No missing values. ZIP prefix converted to a zero-padded 5-character string.
- **Duplicates, first pass:** 261,831 exact duplicate rows removed (1,000,163 → 738,332). Repeated ZIP prefixes were kept, because one prefix can map to several coordinates.
- **City standardization:** trimmed whitespace, lowercased, removed accents, replaced special characters, and collapsed repeated spaces (e.g., `são paulo` and `sao paulo` merged). This reduced unique cities from 8,011 to 5,962 and matches the format of `customers` and `sellers`.
- **Duplicates, second pass:** standardization created 17,838 new exact duplicates, which were removed (738,332 → 720,494).
- **Coordinate outliers:** 33 points outside Brazil's approximate bounds were removed (720,494 → 720,461).
- 27 valid state codes. 5 city values still do not match the standard pattern (likely typos or unusual entries) and were not corrected.
- **`geolocation_prefix`:** one row per ZIP prefix (19,010 rows) with the **median** latitude and longitude, the most frequent city and state, `n_points`, and `n_states`. The median is used because in 85 prefixes the mean is more than 0.5° (about 55 km) away from it. 8 prefixes map to more than one state.

#### 4.6 Sellers

- No missing values or duplicates (`seller_id` is a valid primary key).
- ZIP prefix converted to string. City names standardized (611 → 605 unique cities), creating no new duplicates. 23 of the 27 states are represented.
- Every seller appears in `order_items`. 7 seller ZIP prefixes are not found in `geolocation`, and 4 city values are still not fully standardized (informational).

#### 4.7 Payments

- No missing values at load, no exact duplicates, no duplicate (`order_id`, `payment_sequential`) pairs. Repeated `order_id` values are expected (one order can have several payments).
- **Payment types:** `credit_card` (76,795), `boleto` (19,784), `voucher` (5,775), `debit_card` (1,529), and `not_defined` (3). The 3 `not_defined` records all have value 0 and belong to canceled orders, so they were kept.
- **Zero installments:** 2 credit card records had `payment_installments = 0`, which is not valid. Both have `payment_sequential = 2` with no sequence 1, so their first payment record is missing. The zeros were set to missing, and the column uses the nullable `Int64` type.
- **Zero payment values:** 6 voucher records (5 orders) have value 0. Those orders have other payment records, so the zeros may be part of multi-payment transactions. They were kept.
- **Payment vs. item totals:** after rounding to cents, 303 orders differ by more than 0.01 (264 with higher payments, 39 with lower). The absolute difference has a median of 5.90, a mean of 10.79, and a maximum of 182.81. 88.8% of these orders used more than one installment, compared with 51.5% of all orders (94.3% among orders paying more than the items), so **installment interest** is the likely cause. Nothing was changed.
- The table should be aggregated to one row per order before joining.

#### 4.8 Reviews

- `review_comment_title` (88.34%) and `review_comment_message` (58.70%) are mostly missing. These fields are optional, so they were kept as `NaN`.
- No exact duplicates, but **`review_id` is not unique**: 789 IDs appear 2 to 3 times (1,603 records), identical except for `order_id`. Also, 543 orders have two review IDs and 4 have three. In **100%** of the shared review IDs, all linked orders belong to the same customer: one review was attached to several of that customer's orders. All records were kept.
- `review_creation_date` and `review_answer_timestamp` were converted to `datetime`. `review_creation_date` is the date the survey was **sent** to the customer. `review_score` is within 1 to 5, and no answer timestamp precedes its creation date.
- 768 orders have no review. Review dates end on 2018-08-31.

### 5. Join Validation

The main joins were tested for row multiplication, unmatched keys, and distortion of totals.

| Join | Cardinality | Rows before → after | Result |
|---|---|---|---|
| orders → customers | many-to-one | 99,441 → 99,441 | Safe |
| order_items → orders | many-to-one | 112,650 → 112,650 | Safe |
| order_items → products | many-to-one | 112,650 → 112,650 | Safe |
| order_items → sellers | many-to-one | 112,650 → 112,650 | Safe |
| orders → payments (raw) | one-to-many | 99,441 → 103,887 | Multiplies rows (+4,446) |
| orders → reviews (raw) | one-to-many | 99,441 → 99,992 | Multiplies rows (+551) |
| order_items → payments (raw) | many-to-many | 112,650 → 117,604 | Inflates `price` total by 617,606.61 (about 4.5%) |
| orders → payments (aggregated) | one-to-one | 99,441 → 99,441 | Safe |
| orders → reviews (latest review) | one-to-one | 99,441 → 99,441 | Safe |
| customers → geolocation_prefix | many-to-one | 99,441 → 99,441 | Safe |
| customers → geolocation (raw) | many-to-many | 99,441 → about 10,075,969 (estimated) | Never use raw |

The four many-to-one joins preserve both row counts and the `price` total (13,591,643.70). Unmatched keys are expected: 1 order without a payment, 768 orders without a review, 775 orders without items, and 279 customer records without coordinates.

### 6. Retained Data Quality Issues

These were deliberately left in the data and must be considered during analysis.

| Table | Issue | Count |
|---|---|---:|
| orders | Carrier timestamp before purchase (flagged) | 166 |
| orders | Approval timestamp after carrier timestamp (flagged) | 1,298 |
| orders | `delivered` orders missing customer delivery / carrier / approval date | 8 / 25 / 75 |
| orders | Orders without any item | 775 |
| customers | Customer records with ZIP prefix not found in `geolocation` | 279 |
| sellers | Seller records with ZIP prefix not found in `geolocation` | 7 |
| products | Products in the `"unknown"` category (no descriptive attributes) | 610 |
| products | Products without physical measurements | 2 |
| products | Products with zero weight | 4 |
| order_items | Zero freight values | 383 |
| order_items | `shipping_limit_date` far beyond the order period (up to 2020-04-09) | 4 |
| payments | `not_defined` payment type | 3 |
| payments | Zero-value voucher payments | 6 |
| payments | Missing installments (set from invalid 0) | 2 |
| payments | Orders where payment total differs from item total | 303 |
| reviews | Duplicated `review_id` values (extra rows) | 814 |
| reviews | Orders with more than one review | 547 |
| reviews | Missing comment title / message | 87,656 / 58,247 |
| geolocation | City values not fully standardized | 5 |
| geolocation | ZIP prefixes mapped to more than one state | 8 |

### 7. Rules for the Analysis Stage

1. Join `orders`, `customers`, `order_items`, `products`, and `sellers` directly. These joins are safe.
2. Aggregate `payments` (sum per order) and `reviews` (latest review per order) before joining them to `orders`. For review-level statistics, deduplicate on `review_id`.
3. Join `customers` and `sellers` to `geolocation_prefix`, never to `geolocation_clean`. Expect missing coordinates for 279 customer records and 7 sellers.
4. Never join `order_items` directly to `payments`. Use `order_items` (`price` + `freight_value`) for GMV and seller revenue, and aggregated `payments` for the amount customers paid. Do not mix the two.
5. Use `customer_unique_id`, not `customer_id`, for any customer-level analysis (repeat purchases, lifetime value).
6. Delivery-performance analysis should use only orders with a customer delivery date, and `is_late` / `delivery_delay_days` for lateness. Exclude rows flagged for a step (`flag_*`) from durations of that step.
7. Treat `"unknown"` as its own category, and exclude zero-weight products from weight-based metrics.
8. Load the processed tables with `pd.read_parquet(...)` or DuckDB's `read_parquet(...)`. No type arguments are needed.

### 8. Conclusion

All eight tables were inspected, cleaned, validated, and saved to `Data/Processed/` as Parquet. Cleaning was conservative: only exact duplicates and out-of-Brazil points in `geolocation` were removed, impossible timestamps were set to missing, and ambiguous cases were kept, flagged, and documented. Join behavior is understood, and the aggregation rules above prevent row multiplication and inflated revenue. The data is ready for exploratory analysis.